In [ ]:
#@title Lesson 1 of 26: Modeling, Inference and Learning
from IPython.display import HTML, display
display(HTML('<div style="background:linear-gradient(118deg, #0d0726 0%, #2a0b5e 38%, #8f0f6b 72%, #ff6a1a 100%);border-radius:22px;padding:38px 36px 30px;color:#fff;font-family:\'Avenir Next\', \'Segoe UI\', \'Helvetica Neue\', Arial, sans-serif;box-shadow:0 12px 40px rgba(60,10,90,.35)"><div style="font-size:12px;letter-spacing:.34em;font-weight:700;color:#ffd1ea">BASIC PRINCIPLES OF ARTIFICIAL INTELLIGENCE  ·  LESSON 1 OF 26</div><div style="font-size:42px;font-weight:800;line-height:1.08;margin:14px 0 14px;text-shadow:0 2px 18px rgba(255,80,160,.45)">Modeling, Inference and Learning</div><div style="font-size:16px;line-height:1.55;max-width:680px;color:#f5ecff">How to turn a messy problem into something a computer can solve: the three-step recipe of modeling, inference and learning, the four kinds of model, and one gradient step worked by hand.</div><div style="margin-top:22px"><a href="https://youtu.be/aRdgyZ6lJKg" style="display:inline-block;margin:0 10px 8px 0;padding:9px 18px;border-radius:999px;font-weight:700;font-size:14px;text-decoration:none;background:#fff;color:#4a0d66">▶ Watch the lesson</a><a href="https://www.youtube.com/playlist?list=PLDyCOTShSfCk" style="display:inline-block;margin:0 10px 8px 0;padding:9px 18px;border-radius:999px;font-weight:700;font-size:14px;text-decoration:none;background:rgba(255,255,255,.16);color:#fff;border:1px solid rgba(255,255,255,.45)">All lessons</a><a href="https://colab.research.google.com/github/neural-modules/ai-principles-notebooks/blob/main/01_modeling_inference_and_learning.ipynb" style="display:inline-block;margin:0 10px 8px 0;padding:9px 18px;border-radius:999px;font-weight:700;font-size:14px;text-decoration:none;background:#ffb22e;color:#1a0d00">Open in Colab</a></div></div><div style="background:#141033;color:#ece8ff;border-radius:18px;padding:22px 26px;font-family:\'Avenir Next\', \'Segoe UI\', \'Helvetica Neue\', Arial, sans-serif;font-size:15px;line-height:1.5;border:1px solid #2c2466"><div style="font-size:12px;letter-spacing:.3em;font-weight:700;color:#a87bff;margin-bottom:6px">HOW THIS NOTEBOOK WORKS</div><div style="margin:9px 0"><span style="display:inline-block;min-width:26px;height:26px;line-height:26px;border-radius:50%;text-align:center;font-weight:800;color:#fff;margin-right:10px;background:#ff3d9a">1</span>Choose <b>File &gt; Save a copy in Drive</b> first, so your work is saved to your own account.</div><div style="margin:9px 0"><span style="display:inline-block;min-width:26px;height:26px;line-height:26px;border-radius:50%;text-align:center;font-weight:800;color:#fff;margin-right:10px;background:#a87bff">2</span>Run the set-up cell, then work down the page in order. Later exercises reuse earlier answers.</div><div style="margin:9px 0"><span style="display:inline-block;min-width:26px;height:26px;line-height:26px;border-radius:50%;text-align:center;font-weight:800;color:#fff;margin-right:10px;background:#2ad4ff">3</span>Replace each <code>...</code> with your answer and run the cell. The check runs straight away.</div><div style="margin:9px 0"><span style="display:inline-block;min-width:26px;height:26px;line-height:26px;border-radius:50%;text-align:center;font-weight:800;color:#fff;margin-right:10px;background:#ffb22e">4</span>Stuck? Open the 💡 hint under the exercise. Some exercises have two.</div><div style="margin-top:14px;font-size:14px"><span style="white-space:nowrap;display:inline-block;margin:3px 4px 3px 0;background:#11c48a;color:#03251a;padding:3px 10px;border-radius:999px;font-weight:700">✅ correct</span> <span style="white-space:nowrap;display:inline-block;margin:3px 4px 3px 0;background:#ff4d6d;color:#fff;padding:3px 10px;border-radius:999px;font-weight:700">❌ not yet, with a pointer</span> <span style="white-space:nowrap;display:inline-block;margin:3px 4px 3px 0;background:#ffb22e;color:#2a1700;padding:3px 10px;border-radius:999px;font-weight:700">⏳ still blank</span></div><div style="margin-top:16px;font-size:12px;color:#9d95c9">© 2026 Applied Data Science (<a href="https://www.youtube.com/@neuralmodules" style="color:#ff8fc4">youtube.com/@neuralmodules</a>). All rights reserved. This notebook and the lesson video belong to the channel. They are for personal, non-commercial study. You may run and change your own copy. You may not republish, share, sell or reproduce this notebook, the videos or any part of them without written permission from the channel.</div></div>'))

In [ ]:
#@title Set-up: run this cell first
import base64, contextlib, html as _html, math, sys, zlib
import numpy as np

try:
    import torch
except ImportError:
    torch = None
try:
    import einops
except ImportError:
    get_ipython().run_line_magic("pip", "install -q einops")
    import einops


from IPython.display import HTML, display
import matplotlib.pyplot as plt
from cycler import cycler

PALETTE = {"pink": "#ff3d9a", "cyan": "#2ad4ff", "amber": "#ffb22e", "violet": "#a87bff",
           "mint": "#3dffb4", "orange": "#ff7a3d", "ink": "#ece8ff", "panel": "#141033", "night": "#0c0922"}
plt.rcParams.update({
    "figure.figsize": (6.8, 3.8), "figure.facecolor": "#0c0922", "axes.facecolor": "#141033",
    "savefig.facecolor": "#0c0922", "axes.edgecolor": "#3a3170", "axes.labelcolor": "#ece8ff",
    "axes.titlecolor": "#ffffff", "axes.titleweight": "bold", "axes.titlesize": 12,
    "text.color": "#ece8ff", "xtick.color": "#b9b2e6", "ytick.color": "#b9b2e6",
    "axes.grid": True, "grid.color": "#ffffff", "grid.alpha": 0.08, "axes.spines.top": False,
    "axes.spines.right": False, "legend.frameon": False, "lines.linewidth": 2.4, "image.cmap": "plasma",
    "axes.prop_cycle": cycler(color=["#2ad4ff", "#ff3d9a", "#ffb22e", "#a87bff", "#3dffb4", "#ff7a3d"]),
})

_CARD = {"pass": ("linear-gradient(90deg,#0fae7a,#3dffb4)", "#03251a"),
         "wrong": ("linear-gradient(90deg,#e0245e,#ff6b6b)", "#ffffff"),
         "todo": ("linear-gradient(90deg,#ff9f1c,#ffd166)", "#2a1700")}

def _say(state: str, text: str):
    bg, fg = _CARD[state]
    try:
        display(HTML(f'<div style="background:{bg};color:{fg};padding:10px 16px;border-radius:12px;'
                     f'font-weight:600;font-family:sans-serif;margin:4px 0;max-width:860px">{text}</div>'))
    except Exception:
        print(text)


if sys.version_info < (3, 12):
    print("This notebook uses Python 3.12 syntax. Colab runs 3.12 or newer, so use Colab if a cell fails.")

class NotDone(Exception):
    """Raised when an answer is still the ... placeholder."""

_results: dict[str, str] = {}

def _ready(x):
    if x is ... or x is None:
        raise NotDone
    if isinstance(x, (tuple, list)):
        for v in x:
            _ready(v)
    return x

def _array(x):
    if torch is not None and isinstance(x, torch.Tensor):
        return x.detach().cpu().double().numpy()
    return np.asarray(x, dtype=float)

def close(answer, expected, tol: float = 1e-6, rel: float = 1e-6) -> bool:
    """True when answer matches expected to within tol (absolute) or rel (relative)."""
    _ready(answer)
    a, b = _array(answer), _array(expected)
    return a.shape == b.shape and bool(np.allclose(a, b, atol=tol, rtol=rel))

def same(answer, expected) -> bool:
    """Exact match, for counts, strings and choices."""
    _ready(answer)
    if isinstance(answer, str) and isinstance(expected, str):
        return answer.strip().lower() == expected.strip().lower()
    return answer == expected

@contextlib.contextmanager
def checking(label: str, praise: str = ""):
    try:
        yield
    except NotDone:
        _results[label] = "todo"
        _say("todo", f"⏳ <b>Exercise {label}</b> · replace the ... with your answer and run the cell again.")
    except NameError as e:
        _results[label] = "todo"
        _say("todo", f"⏳ <b>Exercise {label}</b> · run the earlier cells first ({_html.escape(str(e))}).")
    except AssertionError as e:
        _results[label] = "wrong"
        _say("wrong", f"❌ <b>Exercise {label}</b> · {_html.escape(str(e)) or 'not quite yet. Open the hint above and try again.'}")
    except (TypeError, AttributeError) as e:
        if "NoneType" in str(e) or "ellipsis" in str(e):
            _results[label] = "todo"
            _say("todo", f"⏳ <b>Exercise {label}</b> · your code still returns ... or None. Finish it and run the cell again.")
        else:
            _results[label] = "wrong"
            _say("wrong", f"❌ <b>Exercise {label}</b> · your code raised {type(e).__name__}: {_html.escape(str(e))}")
    except Exception as e:
        _results[label] = "wrong"
        _say("wrong", f"❌ <b>Exercise {label}</b> · your code raised {type(e).__name__}: {_html.escape(str(e))}")
    else:
        _results[label] = "pass"
        _say("pass", f"✅ <b>Exercise {label}</b> · correct." + (f" {_html.escape(praise)}" if praise else ""))

_CHECKS: dict = {}

def check(label: str):
    """Run the check for one exercise by hand, for example check("3.6")."""
    fn, praise = _CHECKS[label]
    with checking(label, praise):
        fn()

def _auto_check(result):
    import re
    m = re.match(r"\s*# Exercise (\d+\.\d+)\b", result.info.raw_cell or "")
    if not m or m[1] not in _CHECKS:
        return
    if result.error_in_exec is not None:
        _results[m[1]] = "wrong"
        _say("wrong", f"❌ <b>Exercise {m[1]}</b> · the cell raised an error (see above). Fix it and run the cell again.")
        return
    check(m[1])

_ip = get_ipython()
for _f in list(_ip.events.callbacks["post_run_cell"]):
    if getattr(_f, "__name__", "") == "_auto_check":
        _ip.events.unregister("post_run_cell", _f)
_ip.events.register("post_run_cell", _auto_check)

def progress():
    """One bar per part: how many of its exercises pass so far."""
    parts: dict[str, list[str]] = {}
    for label, state in _results.items():
        parts.setdefault(label.split(".")[0], []).append(state)
    if not parts:
        return _say("todo", "No exercises checked yet. Run an exercise cell to start.")
    rows = []
    for s, states in sorted(parts.items(), key=lambda kv: int(kv[0])):
        done, n = states.count("pass"), len(states)
        rows.append(f'<div style="margin:5px 0;font-family:sans-serif;color:#ece8ff"><span style="display:inline-block;width:64px">Part {s}</span>'
                    f'<span style="display:inline-block;width:320px;height:12px;background:#2a2357;border-radius:6px;vertical-align:middle">'
                    f'<span style="display:block;width:{320 * done // n}px;height:12px;border-radius:6px;'
                    f'background:linear-gradient(90deg,#ff3d9a,#ffb22e)"></span></span> &nbsp;{done} of {n} checked</div>')
    display(HTML('<div style="background:#141033;padding:14px 18px;border-radius:14px">' + "".join(rows) + "</div>"))

_say("pass", f"✅ Ready. Python {sys.version.split()[0]} · NumPy {np.__version__} · "
            f"PyTorch {torch.__version__ if torch is not None else 'missing'}")

# The answer checks. Each runs by itself after you run its exercise cell.
exec(zlib.decompress(base64.b64decode("""
eNqdfXuT20aS5//+FDW8mBDoYXMJEHy1R3cryVrbcV4/JO15Nhg9DDQJsjENAhQANrut0He//GVW4VEstnQ3MW6RIFCVmZWVlW9s
4q1alcnOK/vX3yj6XxFXxyJTvvo35fnqbyo7DOPHg3dF/0ZlVBTRk1cO1KZ6OsQvt2keVf1+/5tvvtlgnGjzLy/e7GIzFn1XL9Wn
z/xlmxfKOw7UQ3+g1irJFN85TKp4X3r6Af3QsIwrGjA6phWeWN70h9HhEGcbz3ugh2nCCzc/dG8+NjdrtOh+A2yRH6u4FHgHqqyi
ohqoXR6lDfADRfcQBi3EMAH/jDF2OaY4RNUdTZSXVQuLZKuO6uVLHrC5iv/RkDWA1fGQxh4G6OsR+p17Ber6Ekj4oKlHIC2PN92h
adIHleUVfseg3V/xP4L4QSCmpV0+3Mis9HktE9PvmhJL/pduGHUIWOZFFW88QmKg7uOnl2m0v91EqrhWxdK/qTmhyo/FKo0zLy82
cTFQ32vSlFV+KImiy9FAfat/G920J2Ce8srj3vt+Gd0sb28Y62igboHVn8nB4zEGMtTSv75pMeBtQWvqmckymggwfN9XV8pvT7JP
Mq+B8QD4eJoD5iCOLKo8T8vhIS72xyqqkjwrvSLKdrHnD2jYvymfJ+U513fx+n7lr3zDxFFZ0gCqjPaxoF+uJgPlByNa496e2DQ5
pE+KLoUDNR6oQEXZRvm9b1Zvfnz75n+/X77wh/6LG4Lda8YeqBc0gALI5VC9olXbHwjVQvEdpYrSVOVbVd3Fe6Bwiun7MaO5VUTI
pmlSxus82wxf1KQyQwcGbOCf4WHgSKARgDOCbtTemi3MsuN+Jdh5GSG2J5YabqN1lRdJlNIlurbttW76lH3uq/IuP6YbdRsr+vqX
DsqBhXJAKJ9DO3YReZPskqpcBcRTWM6yKjwLHMKiD4h66/yYVSCTkodAtGDUhWRsQUJr9CIYhONgsBgFg9FoPvBn08E0HNHnkRL0
BkRnQvaWCH5KaHP5Cz3BtYpuIUSCYag+HpOswmLkmWMhQgu1dZqXsfcUR4Wgdo4SiWg/XkBQj6cEybcqCOnPeDoZBhNCtojTl358
NQXesvyl2iZFSTuXCJApHrqDeWhhTlzw4hWDP5vJ7QPZjepQ5Lcp8VqEJSVg6fuuiPYkHtKHuAQXCffdRekWNwn3qffxGpuJNgOe
O5WK/jhIMXGSwuzXXXXngUcD3j7gVMLwe/kDueLf0Aalj7hFf8St+iMe0R/xqP44gRgCoZJsnR43MbNIVSQHdRut72nfCc/Eh7zq
fQGyZmMTKAayL8GOqSO+i6UBOLOIH4ixYiLhg4ADXlcyRmfVJtaqTZw7Z2poqqWuDITDrYijzZN3G5cVi0QP4JzjKPcPahk70ftp
f4iKWEW7KMnoIAHMTyrPYi2NVC21emfb1hwlgAfnaocsvBgYgH9W+yMN/pCUiZkC0p/mWccybprTV+exsk12tDkf6cdDWg3L4+0h
zavSo8tl8mf80guHU5pxGBicH4e4wTtU5WSJUTEQnZjNV58PzTQvXv726ue3Hz68XfYibPweXf+T534Z1GOV66giMS3D+detwfAF
Q5Uvg+nobMT1U5S1Bhy7Bxy1xhuZ4cbB+XCHJLvHcPuouKfhemXPHhrbOhko73GgnvqsnZE4i4uIlhrDtw+Bx2EVP1Z8J07hwkto
re6il711nBFoNPRD+4uA0vsfo/VoEQR04RQnu7vqZe82Tze9BrGY2C+pSB3a9j6A2yFXiCeZe64Nu36Sf6+HwfazfpaXleSI129v
iqm1KWiRX3y4i0g1KhVzMDBexzRwYti2ukuyHW+/+zg+MPdiUwzVT5U65QWdsiDSROswuG8bJWlJvEeP3cb0Y0xniUOczVyHVgG1
YhX454LdJ8H+bw5xjx1xKkg7UYH/FxqL/lFVsidxa59gMwt5OsdfvI3WdyqLT7JztBqS0MPAE+ip2yf+rI8x2r+M6FAx2XZFfiL6
E/VO+LqP7uMOIUmSH+I0LksH/nMX/tuI5E1ZxVDlSEZ8S6eX4v++VaIp0cXy45GEy8ZgacRwMHLIYBlPjjnHiHJW8lGIOel0fkg2
NDQhHalbOZQ7NJxbNJzXZ2FIh/tenlAaAeKL0TAMVD2/kYfm3DyjStDSF9lKuj0m6QaGRkQ7Z/3kvfnpw3/3L8lMKP+9H3sQkkuv
9xttqpA+er3f6dO0z7LzR9qSdIjkEZ36RLfflBf2mWl/V96033t25N/NyK8S7OAFj/0jj41PmG/Mn97Rp0Dm+53n2+IE40m/IxJv
1FN+JPyIzDkxzyYp5Pwv/9f5/NDcaPK+HIks+tdJ9cSjTsio2cRtXSWwVOSAVeRzIgdOTYJNvxUsH2/JeDFKQHugmO6k+gKIkJSD
ce8LzwuNmBTnIwj9d7kQ4BQ9kUJIx8i9Oh4ULLFjnxYn7yAWWIgFTsTGTsTuPtLBNyWoA/qP6Vh/6cwxtuYYO+eoNdJdDiuYbAxj
NLNBrPEUAoIKwq6nKGObWd+Ku85vsleeZuiL9o7HxXjosMCn+rfPSkYmUUCCKIPohq7EukHEjHKRuWUW/cVMVOswMSQkj13rYFji
zmhEAy8iQ1f9nXYs/rWNU5oCXoRKDFOIG6jHJGUw1ADmWnTAqcaKeGdNQmtNwtaJlWTbuCCxEGMknFh0UvG5lEClh7aVqZmmy4Ch
TyNYhxDItN1YXBNCCkLSIYomLgHNCiHobzY/Sxfhc15Is01xo6bbLud1oS+7OxI0AOSdY//sooMeGQpyzx/X8tIPOzSZWDSBakt3
k/kumPr6I8EArQgskREQOMcNFL858J1e2DykoUdFFm+IR45778O7n357v/rxd9hZ4L76OyvrJNSMjSDKgJwoWCBSRojiGR+n0I2L
uIPU1EJq6tx8s0sw0hlOKzIZTmhXT/XfEVwS/pBNwpDBo10R7WL2C7BMZjjPNwYNtsL6Der9+unbb2XL1ovev1Yzsig/d3bxcnSz
FIuJ7d0fccbQgld8a7NAIJGwRr0eZCBnXVE+swgCdeVnrATWcX0Hv8uGbQpRTYTBmy1B+NH3iJWbKCtP2gbHUU2WJ42RxnoUACnb
ehNrvaaMae+zhwd7CeAeiH0qY77QgZUcYgcLOXWack06IN2SJ+sYvh6aJqX5eyDThkbaACHWn9iew90ljMo4A34AIFLyNPHPJmb7
5lrVw5zNd0gIliw/sd2qycGTCRlYnVUfj7S+wI8XChPzgAPMKLbUffxEanx+TzC3ZU3PjV+52hb5fgUbgGZK9TrxtLI+WjKTWKW1
oVvVJqqi1tf6kTYPzC0emDs2xfjMxyabQshOK6w37onUvgNLZqLMwUjmPwbqtx9/6vdpq7wGsIci3xzXVdcnwvIsofNGTMsawLGl
b4xZ33gla8hOpGt+Nt6TSUDCF76WXIYrD/CLkGgYnuMTXMbHI2AHioAGsMFwZKQthEz+otQzaR4KepdGYRtRzEQ91ut608rqb+Oo
OkJMVerPuMgHDW+aHQGCXJ7AF8NzOR7SPFckhOAzHk4YalZBtNOPbQw+LkkyNkaGnr7sEDuwiB04ucHpDCScNsm68jTJQP2eKL6d
nwEosyypNnIPWCInJSJ5YMRxcUAyZReZK5nl++mMN3IMFzXsET+SJZc+qZEeSJmpOmiPLbTHTrTdjsIy2e3zZKMBkQVoQzAyztcd
0IE0hW+ud3kgIWAdB2J7lCNBQb9vjR10xhaH52g4n3fJpce9muCc+jvdMPJlXfQPfP1/0vXFgoZn3qzJr/kcG4suRgV4mu+ol4yv
+h1yhhY5oVFhssijiWgdzKIYYlxrx/CDCOoUihUbvDjFa8GJSyTCSFkpiW8dm9rtvSTDsKLpVoyJbBS2com03tW4T4Be+b3Lj8Hh
6loNXxZD0xCLceXzQaIXIZh1SDKxSDJhJdPs900BzwYLaTa2r3w5a/lYJAAiMpShkPMdMsN8/lfcqZ3cs78iOHFPR1KmNOSId7HH
p/Y0bFRvW8Rxz0G6qfNUpbOJDm/I6KU8iXMnye7hxep1JuvdLOnw9hCrLGidH/nTben9QVS6aev6LOOSP2MJAPB+YUyvxh1yTS1y
TZ0b0unfgRRYZfSprC2OTGIspFqRxbBnL9vb/3z108/vTRAUATxA8i0d23/D4uHjXl3RatC+4IWOfbMXsDhxYL75dHMH9JkF+oxX
mqh3di6RxD/l5moOndWxMnMnU99C9YqKpxUci6UHjp6LsHX+DtW0Psh4exXHTDOcN+JniZe8EHfJXnhEsJAWkW4meNcx7oJ7WAcl
v+TcnQwDy7m7TciKJWvhFJNG/0hrs7RgjPo6xEGr81jSWTa/6E2N0sNd9HI09Cdd3/FXDet2IbvhRKRidsMH+ddDw7xG/HeZ3Rqf
brkmEro5r+33Nc7nZYQDf8/uZn96yd0MbqaBiXNVnJax+oJzW88RZXQ8wu287X0C+J+v1adyff039vgaXPAvw+mTANtriPsdX/Jj
muzBLbP+d+bSk740t26MbuPU0xKlfbdcjx/XabQXE4HX0uWzZpc1VPwSt5m1v242GZ2QEfyNTJrLjuuxpQiPLyjCC6fIgR5VwaH8
6MlKd2L0GWKRtEl5meX3ZQYrrlnx5qp/g8WXwyWeNFJmNhlonXAHkQymwlbriJ6FhcLCrcuP3HpMFd2m8arWFvzRyKg0o7Zose6r
b/MbVZkslCKG57itnYwk3N6YiGQFFUVe9OpN88BB8CvRZ4laPLXo4GehcCcsD2696aEvfi0NDf2wFnvsWB4jmOnF/phG0MM/PXzu
2h4j2/gYwURObouogAtfZwPoU000geoOqkv0NCSRWqzvhua6OMxy+rNNEYEFxnV0sSC9xhGcDi8YXhs4u8aiakKR8dkdMWrBHlp2
U8h20xhExTJEm434aozVBfWFBsFKlS1DhHVM9hE5YAtce0Eyax7DFe27Q0wA/iMcmo89KHxFflIHogTbUQPWAUmKHfcZX9VWSQeT
wMLEZZSEFxyzsn9IjC8DZqsQf5kM4LAbA9Qm51A67FIGo4Btbxum7KjsQDa2IBs7IQtddIIz0Wvldm3TaFeGYNTlh+JIh8d/RCS7
B6r1RUKzUBnER8L+JS0goFs5TJvQ0sVD7d2MxfjWSmZCqiEvh4wD2QkvTazDKgOtd3b1y0FtG+QnCCQCKkpTB5s4nZyIg62IC+FL
DMTlX2BXSMRprFmi7KAysVCZOKk9vcyVzIek0ccbHGREfNjdpeeNEf7u99W/q/oafe2yreyHmnFJpJMdhCAhEB8HHUCnFqBTJ6BO
7bVixaUi8pzDN+iARxjQdT4n9A2aeN540Ncbme/ofeUkNERwPklPhLRMw7e0p0GEIMkyoooo9WNJqhp/5ZyhjVdoTRnpoHWL1mzM
wDAxoWkt1hET7qzCzFqFmTiOsKhy8sBco139xNsIfiI49nWQWe1pf8J5r+PLSQUZTUOVcj8xPnFw6WB3W2En3HhOLYpAwJcvZWX0
cal9pxoPTjtxcpyAfr6mbUkHTmArDDJleVoR3YitgR1iYMtrUkygdrzm4+ceB66k140RTWSA6HtJkiQ27rlSYr/tVYaQGJuFblF8
blF87uT7xeUNSpMf6oPDK0PA+wvRAiCTcGx2JNsxvPxkdUeP8J6WSImI6WhH3DpXYQeyhQUZNKPXiBGuI47p6NMZ53mKgIiC64H5
4eMxieGr4DAxGCNEpCFUrHkM1OkuWd9pB3K+30MHPcLVTCdHZM61fRxXuKFw5FyF5+oYqbIZ8V62rv2+rEgMtD7xIc7IrHXzzi1d
UBXfcJFDzGj/CIkxhEG0Y5Y1EeNKpo39y3H/21NXpxjZSsXIscQTd0pmJCHo1XuJcPckCOX1XnMUvT7cJHKbxtgG76+xlK+YWcVn
RV+Fd9vrO7GUnYkzKD1xKyzZClm5pQRBTTDm3a+vvic4e+/h6fihJ+p4k7EiibzncdCyDoS2YAss2AInbOPLu4IDgjpX2ROQQD0m
XO+NgdDE7hHo1ETEPn1/9WqgXl29Zuq9vnrj4AsJOHKYlGPnvg5d+xKFb6EytlCBvkNkohleD9SbgfqBl2ZmwlCiRGgW5T1Sxh+P
CHEMOpFeHYqpPfenuzyN63uH58QKL3j6CA8WZ0Qnkva+nCasvMwQ+Cfm8UMDXAex0EIsdK6R28HImhzPXearbWRS6iSfbsZ87Wtl
kimrPxN9609dKk8sYCZOYNyxVP670avJaTcdhpVQILEBlmNHktURmSTLGpHYwGxI3h506HOkghOV4pPEniVK34J8akE+bfPHD7zm
8pV5hVMzeP0XQ/WGAIcgFhMh2tC5lJTxps4Gkxv1WtJCLhxs4Q7dlic6VIi/opQD3yHZ18LmNSs0WUMdbGYWNjPjs4Wc3MKmpcNi
LVm3kdokW47fVTpzkuGtTogpkizdH0l0q59wYwkvy9VtBOxMRFByLdnJUzoQm7tVxThepbS86YqPZUkOYcnV16Jr3KR3Yj8smBnf
s9YWiIzQLDhTZNJmUG8Mpy56/z8TjnjCG4m7Hmh5Oe7yLySRslGJ0oYOjecWjedOXq9VhvQByaZLmuvmpvYbrFpqTMtTIPcepN6i
qTKgy7qcg59r4bA8LK/8Gz0wy6OX6g3y1pH2iZ/ag5yn0+CJVVStiNXxcUny+aab+m5oDAFHClaKWAaOC9DotWsnCn/pzCzZyAnA
KXU6t+RyHw+9r3fKjmunLPvI6WbtJAfMtYPtNio87UNfMjLZTVOnwD/UntSlw/uYQcUl/C3/I223qnc2UMerZ5x/NalodogF6GMt
H6F2/vmLhqYknJg3IXgnemc+4/CbWDrhBDphe0C4JlAuooeSswxbgFMyS07pqYM0BBNsEy0PiM2f6O8e9nA79R4lMDpIT5QQvZq4
o0ji4gUCn3G8uToehj2L/6dufSpbYQGOyDbQ2subX3/+9b/eve8j8xLf37394adff3nPp+C4Nq2tf1vbcWrpUVOnHjV161FJuXqI
0mTjfSLt5Fr1yMoWJYU+74hsmWgr9O02PXIo6Xtz2+eB+s9Xv63efv/DW0gSuDxEnUlKVeNICxNH9wjiquKYOg6u5wDQn8+nF8gs
ANjbAgheidYEywuBejqxWFCzB01D1iFfYJEvcJLPqephvRj8vmR7tsRGs8zCOwd6TuuhAtBZplzLhCA5hzi5eHDx28MyulF/eake
7BKvhgKNN9ZApIVPsxiJaGqsEjfGB9/eocjYogg0xl+xr6bGlp/7bQR5kc0aD89p5/Sl4d7NikvjkIbiM/xrq6RMvHtmj6BK5xBH
1cuQg39reOMhs8Ao4uyGJ95c4R30hq9/r9aReALZ2sqe2P+jUSg7uIcW7lzVA/cZGaoRcp2QkFqKmNGq1UNUJDAsSzgc6iAmlAbU
9xQ8N4LFatEmWrTlnEMajTOPKhzu8Q7GloOCk4veyPUdQYUDXNfQIL9ERAfo+QoVc/jwWqsIhh7YGpyKvjnfk+yEB+nrwafGUDHu
1oZ7cHUqPNTCrkPSiUXSiXODOV2AvJ+0hNCCsbXrSSXVwrMFokn52ZNFBZ1p2nt21KUxy25qC5e+9G/OxyatOoNf6xY8c13L4uD/
YfzOqIs6tSjLtUwgxjnbs6Vjg04tik6dFHVXUtCgX7HjjGZmbTnIovUyYVm0XnoJF3eqv6JIjGtiGqUu7NQQ1tl0kvLOJ4Ng3MFr
ZuE1c+LlziY8bvL7I1Jh9G7Ursj3fF0U2o2kRnISi0CTkIFBqkHi2AVN/j2XQZD63+tWWUEUsm4g6ZJlk77pyAUkEjLhbfD0D1BC
MgeQpHMqdlFIWo0WOKReOZiO1J9yteMYdgteFnr7/CHugIkL+mDa5mmanzrLMLeWwaXfzxr9hhXsgfro5qRlKyZyU7NTcKFQNtlz
nQ2qjD/Cx5xXyjuwzMLHjxITNDd9OpAO8OnjZ85PYvdyC42ZpRfNnHrRzK0Xcf58XqSbkpM0WJ4yTfdkK2KXgqoVYYRili0QPF8Q
eX6JDD9P06D9T7+VZ6dvvUJqoCfBo+av5I8hNRmPGQ8vLjAAMEg6aAcW2oET7Yv6DM7lFZu4wD2UMI8ASDdVSbl9EicAbuxmOnpn
QbC+KbBvjYpCjyjRGbZkgpO4I+UcWclaWazqSJfojC3cxhZu0EzCWimpwTwWD2Tps+lep+nAsVMirRaDSmFbfh+fh0hnbnUF0K4A
N46JsyXSMcn0yaRtIYUT+MkTp7gyOIoaWqbJ4QBBz14TDW/ZQTW0UA2dy+hUDIhHERf0XICKj8ho6y0Ya5CQ68cDuHSDZmgXS7u4
V89Yq+fwFTUE0nnnHXIwAOzzInPrNs3X9yRWOrSZWLSBRvG+BT9vjMRkbMtsJYrJEW4tEw4C6BAJigYNwjSNWIwku4inkIRdGjbd
HSM62Ko4djCMOxGP4Fnx6ScuHZIgmk9D8SCVEjWysO85ah+TbCVSumzRkfdQC1d9ZJha4n2HYFOLYFAYxma7uMDQrl7mjXhTj+8j
Y9HQsM3nrsdhEABIXsJSW0WoGC9dNf0zt76S5XrXQfWAMNlD68eRs7fkCvsw+AZfomI1A0gm6pm8GQBB1N3hKUzSIdnMItnMuf9q
XeTdf/389j18V17vHYai+f+Iqx57//EBvjZNIdIuzws88+IUFZsVa9xkDPMYdLrxsDTIp86o7cE+A1X8qMPH9Ls46v+A0OFL9b1f
nLYZtT11dy6W+8IOxZGzNDEEacT1DpRZiap3zkKMS7iyPaML+Und5+n7X8a9q5HdiYCv05iYI9hH21ndubW6bhVn4Xbh6ApY2teS
lfzPYNFkEU/GM1PwSpLvn+PGmYo0Imf97Mxya8041ElPNioHsT93HpC0LMdwnIkxVD93KtlIjj20N11Zy1yURptkIo6KcqcdmwJz
txOL43zwcC5G4UL7qXJdDoSokd9OappbetjcqYfNLyQmwXvN5azInICLzPfRnoaLvgNRoXcxyemqSNawaq4lAEb/Lei/4XDYgSSw
IAnqMmUITjmauQmHmHYSsAauOgrAnkTWmlEhlooDk3VB9mJKsh97iMWreZdDvu7zsi6RYlwcdB67rV9jT3NPk3FT7i2sNh5OHNw0
t7SkuTPDaX7BK0NctBojgQ+TCX8/z7xzS1OZOzWVeaOp6JJX5CnnO3/kSS0M/eHlbT4GkkYocNy4mmzkO46ZEW9weSpkAbI/eFiu
I8HKtPOu+O6v87tPh2Hb706SbsetmHpFvE3jx54xsiQm1GtZds0VAiRZ6683LS/9ncfDDVQLA9svL273wXkecH3hIcnTuHuPOPNv
Ok0qHrr9KeopHU0qHiRLeIDHtj1/9M9PD9dDnzOJW00q3PnAhuamQcK53597KgBxRHUZBNoZCBBcdPfPLUVv7nQdzS9EVct43dk3
TVOD8T/pm+kFVbVkKPJ2JH2ud6m5UGgGDEfPdhQS30f2wDtL9xMiIWm3EZpbqtlcVDPa2HUTIlpGAlf7Bgf6lBnPJzKW5FWMhur7
/HibGpFuOg7VPSH0qjgkz+xyDoO0seCkNenRUycsNBVpUoHZZMYizoyM+Q6OMwvHmXMZnX6dw11e5dA6ZNsZCyLJDkcYlpKLykyI
E8zlzPzzaxwiGfE/hnJU/Z5izru1fTe1tSdOJOPIqX04PafH+zbP741k6Fn5QJxsIaYHh8y1gd2i49yio0trWbjPbCnl0sntyTW2
uC/mvT9q54/g6Ayv63YUnQ5P+QGuJ0c7CntwnwcfmQKWk8nJxQiwvGOyZSS4mldVvh80td8wGkjI7Nt4LywNYuHUIBZuDYJ0glX5
0ahqwDLUyoE/7cwRWHMEzjncCcvEeVIb1Co95fzjaLNhb55VHyxPJZk8NOZEcBIqktHleJgFE8jSgXhsQcyt1+q0iCtfbNnbIlrf
cy6bruTSjCuiAqpMnKayCeDijg5Rca4ILi7VUX5ccVYmQs6oiuCKggEXSNQ9qeQGnl4KBK4kKwxfOviEFj6hcwXc+Tx7+s/H4P9Y
/fz21btfBuq/5UNT3olU+VYjG1erEozC5QuOUbyQeQbbZILOAWzTE/lOKAYyfnukKSdrUyA/6qA3sdDDWfZKngdorDqSZS0JrlJj
OWj3GGKZUEc4HEv0TBJ1vD9UT0ACja/EoUhKAS5CaWZYB6q+ZKI8dNnvoDC1UJg6V8id0aMHXeVbb6k7l4FFglDqIboB7rD3/AB4
0hf13wDLCqqRHuegzyzQZ07Q526DT0q9cUaYssQm0OGjwG40asoRv5VMChTYAEJujyVdNo5oaChnBvp2tqtqMUMH3rkFL0T9f4AD
/MFoNqEB0EkDJqa4VXSDyxI5DJJRiEBye4aReAV0fClK18dUyrQkEFnp5inFEQ6fIU3jk04j09DlQ6SzgdDY1cF7C3eWHSmbPo1D
i+WzhsShRbrqyZ6s8qbpoky4zg+J1JnSXcqWEAuLKMj+EPvtajYNuE4GonIbS6VyQed2hKJ9UFtaWLxAKOXJOOC4yp79QpitKdoF
CFHKdhvTi6MvZykei0tVWWtabdQgb6SyaRwYrUX2NGehMZ/3LjxL0AwaWukxOC0yYXv+ORpZ6b4LSff9lXUmhUxzk8SJh2uvxdV4
GM6G6j2y5JIM1VXMHAPjyNuA11nhY9cZHx45c+15C7PRhQIoLRw7cppTz/CdjcBQV/ewPGRBwNoaK3u9ywO2RXafT6KRLkOjPzed
kgSu7dfPlai1rJe33c9sZPeUHTlVDrrs7klxWiGmiMNwtqi5fGEORPymG6egDj/WO0t0IWND3eantAtSYIMUuEEaXwAJeXwkbycw
W0IDC6Q/zfioyc3e80BXLda/GSsCQjXsAjW2gWIlZCLS2A+NuUKEmE+vL6Dp4iC3urEros3FA/pqMjLHMiszfA6w1hFC55jqknXI
Qen34zLyeAZNowvKxOh8Rcs0P7CwxYYwUdovrWdoky5sFIKR6gyr96g/HdIe/aVu84Cfr6GioxKRu+VwXVdR1sqEi7QX+sWCXITa
UqrtMBf+GZsdFLR8YlKI03w/359iuh1JsVzdRcWGC20D8clv4kedTm2K2B6j/SG19t/EJs7Ezexu0//k80KhSLdhihFnDpfyQwM8
U7HnOrvKkoaRDtvZYbiPo8zzOqPSB80h6qpmEXY29GtLmVW6Vg62yKEuslMbWehVJx2kZ7Vv1LSNGA3n46bRBA/f6kThT5o7w+F4
4mKAWj/j/PK6py1YP4mzakXLBVePZyTohFsIuFvR6QqHCStjul94nTSsDulRlFktfdm/UsUHlyyngZb+jWPdmnAwHpVKYFPRdcch
9GdG5JB6vZ/1vpWmX6z2TEbSlZK2Tqsc5/IGfsZ12G1rcJITAE9d6Fer/V4n8YVK8RfDfO0HN1Z/3RNtzJoHT8/wnTRy4pb2pxto
HVe9yw149+XLqdObd3JU9oPTzp16P2imUZpp1ClK74kjUXNaU+2Z1rMju/3qSEr+zhYnrirtb+ezRJe68q/lXZFk9zqnJyEl5UR6
y228jo5l3JKj2xRhoKyU5jfNAru2iNMOiKrVKKjdejtkKosJSGezHu+YcSoxA9RzDzBBmn1yNsCkMwI8by6ZVKzSZJ9AmMLtGMx5
j7CrkQ/uuZy74654mdsUnstBA7BP9ZQsPrj5QpXLv7y3NN2JhKf0aaheaWAll5IfZY8OGhzojoImBkI3Z6ZN8G3eanUU+KG0MolQ
ce+g/uLi4b+KjvAGto76Tn8Algm4TxnDvEuJhU2JRbs6BGfedIbCDx5HtxgQTmP/oITVmmbs6renD6jL0w2POfUZKJq+XWxlOPXj
0QUdreJuI17/62QWe3VFsraEV6NwWSrt6EynHZmumdKsu9MZgfeMEa+1Vwm8x4mwXQPJNHvKlM7/PhZl7MT8smkA/QcO9JaGKrjM
hrbW6Z+r5079/MJ7JnTdal20+qGpUUVX4qxCBVsJPz8aMJUAq2m8djbYisyhPQbBv3JgIUwYPXact/7Zeyr81QWYg0vFyHR8rgx4
UpAc6Aytdo8Gc8elNg11TTI/7Do4W9PQubPUacV4QUBdmYT/39y0m2eKQgfMuWsVl1uYGJwZrUuPwKZH4KbHM1WWnPTXWkc+IFFa
/DJoFR7fsnAALzNoOt2qC8zYBmbMpQQBuwp1iJx9JCy5TBjd1B5Xxqx+oRMRSU2X9AeTdI8umuKekepjrn8GuBX3uDxzLviX3nHR
Xh1umbXpIi7uPUYUMySlrtCT1AgIizr3ibmbTaTGImrTJLRpEroXaHKJYXEarZj6un5esytDxz3hdTvFeknOXev1ELxPj3vBUnxr
sFCN47ret0jmQpXFk3zoojSxUTI93bpDME3Y/S79OVAi/lR/HjpoML3gADuK8UMrFD0azqxtg6jYxU1uQCNl5MUh7fD1VBari8zU
RmbqXp/ZpfXhMVesXNcLFLTWiNQooXzd4eX8zGqGYEFBIuKDvEykXuiA2+Nr2irfKh6QVjqdDGT/rCm/7srPRw3vHLwTCtFP7h+F
fhLJRpddZGbf4aUgm+M63pgqGsRnTaW8SRF1UMup+xXxlrTJstUGg+nUb+VSRiq4YtgkZXEdZS8qrZdEaiw/HTgTKetdHP9DO1Xy
AxcwNMcJIwL506XV3KYV63cajDrzXtpvNIFbHlA3zqjk2EaTuYFpnhF1n0ARS3mApeii2eJyO4SOvHqmKYI05zDMgWq859oiuEY9
l4Jfao5AxFvYxHP13/KDCy7N+/hgSrYXTVvu0XXzUoxFezq7Eb/v7sTvBxdimQhiS05DMOq+/0E0pah51QIifuZ9F10QAhuEJgsq
8FvvaoikPbpGynolk36T09ABujuVifZMCVl+KYyC7SRBlEx9y/8FEkr5u+NtWXZzSghvTrUtG8GZdbEe21iPTUBlrneBPiLlqEfb
7cbLsosOdGUT65O/9VYdFwHc5/ZdnG5W91Fx8PgdR903A3WsmPaLQjRQkyt+EYmuO+09N/qSX27TKGv6VTdaixux0haagibG/Frp
Jwb1+5tGOkHH1x0lgi4xQ5uYYduO0ukMUSk5NmwAkvStX25Ut6bSiYdbNdepjk5+qrULkpHwVmnCzR0ZYXf3c1SYbIGe+IcMWeqG
Q4kp/m2/McfVXwOzNKO11+RsKCDNb6M6f9+U3ZXfl7b8r3FmRahjHarXraH5JjqxwtFgHJjXpX3X4kt0usSJJkYev1PtMuWmF0p+
+cVVKBjUbDJunWLOF0Y98/xYP18fWvzo2DRRYPuRLgTSc+WZ4Vqw1GNF8nCjx3ZJO7VJO3ULU2eKk0faHE1dRekAit3KVM16HMKd
nL1paCJiZ2JSXXVTvomqb+kCN7OBm7mBm1+U9E1v/FtdLbaOisK8hUi/uEusCfaRHlmVTnYZOgZ9p9bgqV1cGYtDn/UFclq3VRfY
uQ2sK7PIH19ILULzXiT2nadp6f7N+LhNHuONdjFwbLHnHuiYccc6XR3/R59LDLWOIM6OUvzKTbP1BhO787w/dh+wY/cBi6ZPxGUG
mefyvYqcRFh92OgHn8v+MmPrhDn0O0C+ayBMheHKjtbvB2ZUC8PAxrB1fodhfYDT3MQKkjyuVTpdgaErmZEefsz4qsMHeqFhPMO5
yu89fcqMuGx2pNtC+eYFR0aaGAx0I9FAbXJoxVAE0uhgDNEn2mtPdb0+pug9P7E502Tim7bIsGb0zVwiingKXrpRl6hjm6hjN9uE
7gQRaFosoMt2bW3xlbW1k7q2ttC1tcXyXzcmgfdfGOXX//P23c+vfntvv5iTcTE5mKaYeJ1G5V0Xv9DGL3Tj56wne4h2x3h1Hz9x
AK/XHBNSWMe9qaUxiUk24dd2JBU7OCz2ndiQ4DzsvecXXhgnm866b7WRzw9xZmqz0DhYd8RuOVTYTLpu2qaj9/nAekcbdy3WzVR1
Y0wyIjaOHt7+hfbqa+64vBIVBzK513S225FYkJhd85YSW/SaMo4Nt3Fq02Vq02XqXiF3VTe/Y8vo7OBAKeKPC+JFkg/3fbt3IDx6
437n7Tx8osQb88YoOO94lY1Wc17UXNVT+vXb40Si+YH9Hjo/6OI7s/Gd1WLMH47rlGxjf+iQ5kLB8VZGeD/qW90ki+BBqiqbXEnd
rMl+a1/7TX1uhWl84SyWVjtchscpufK2t6+o8uKA99aU0QkYugwUBctx9R32/dlVvbGlErL86totU0bVAfQLtVPSFrNdLyUlQ3Up
F2vq77q1cb7dd9uvG2//XxKmByk=
""")).decode(), globals())


Every method in this course is built from three steps: write the problem down as a model, answer questions with the model, and fill in the model's numbers from data. This notebook walks through all three in code, on the same examples as the video: a courier with a list of stops, a spam filter, a small road network, a coloured map, a few rules about rain, and a line fitted to three points.

You'll work through the same ideas as the video, in code:

1. The courier's problem
2. The recipe: modeling, inference and learning
3. Reflex models
4. Vectors, matrices and shapes
5. State-based models
6. Variable-based models
7. Logic-based models
8. The ladder of models
9. Sums and products
10. Learning a weight
11. Named axes
12. The courier, solved
13. Writing a model down

Every section starts with a short explanation and a worked example you can run. Then come exercises that build on each other. Work them in order.

In [ ]:
#@title Helpers and imports
import itertools
import math
import time
from collections import Counter

import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
torch.manual_seed(0)


def draw_graph(edges, pos, path=None, title="", node_colour="#a87bff"):
    """Draw a small road network. edges maps (u, v) to a cost; pos maps a node to (x, y)."""
    on_path = set()
    if path:
        on_path = {frozenset(p) for p in zip(path, path[1:])}
    fig, ax = plt.subplots(figsize=(6.4, 3.6))
    for (u, v), c in edges.items():
        (x1, y1), (x2, y2) = pos[u], pos[v]
        hot = frozenset((u, v)) in on_path
        ax.plot([x1, x2], [y1, y2], color="#ffb22e" if hot else "#5a4fa0", lw=4 if hot else 2, zorder=1)
        ax.text((x1 + x2) / 2, (y1 + y2) / 2, f"{c:g}", ha="center", va="center", fontsize=10,
                color="#ece8ff", bbox=dict(fc="#141033", ec="none", pad=1.5), zorder=2)
    for n, (x, y) in pos.items():
        ax.scatter([x], [y], s=650, color=node_colour, zorder=3)
        ax.text(x, y, n, ha="center", va="center", color="#0c0922", weight="bold", fontsize=10, zorder=4)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    ax.set_title(title)
    plt.show()



## <font color="#ff2e88">The courier&#x27;s problem</font>

A courier leaves the depot with 5 parcels. She visits 5 addresses, in any order she likes, and comes back. One complete round is called a **tour**: an order that visits every stop once and returns to the depot.

How many tours are there? She has 5 choices for the first stop, then 4 for the second, then 3, 2 and 1. That product, $5 \times 4 \times 3 \times 2 \times 1$, is written $5!$ and read "5 factorial".

Below, the depot is point 0 and the stops are 1 to 5. The table `D5` holds the distance between every pair of points: `D5[i, j]` is the distance from point `i` to point `j`. A table like this is called a **distance matrix**.

In [ ]:
pts5 = np.random.default_rng(7).uniform(0, 10, size=(6, 2))
D5 = np.sqrt(((pts5[:, None, :] - pts5[None, :, :]) ** 2).sum(axis=-1))

fig, ax = plt.subplots(figsize=(4.6, 4.2))
ax.scatter(pts5[1:, 0], pts5[1:, 1], s=260, color=PALETTE["cyan"], zorder=3)
ax.scatter(pts5[0, 0], pts5[0, 1], s=320, color=PALETTE["pink"], marker="s", zorder=3)
for i, (x, y) in enumerate(pts5):
    ax.text(x, y, str(i), ha="center", va="center", color="#0c0922", weight="bold")
ax.set_title("The depot (0, pink) and 5 stops")
plt.show()
print("Distance from the depot to stop 1:", round(D5[0, 1], 3))



<font size="4"><b>Orders for 5 stops</b></font>

How many orders can the courier visit 5 stops in? Use `math.factorial`.

<details><summary><font color="#ff2e88"><b>💡 Hint</b></font></summary>

5 choices for the first stop, then 4, then 3, then 2, then 1.

</details>

In [ ]:
# Exercise 1.1: running this cell checks your answer
orders_5 = ...  # your answer



<font size="4"><b>Count orders with a loop</b></font>

Write `num_orders(n)` without `math.factorial`: start from 1 and multiply by every whole number from 1 to `n`.

<details><summary><font color="#ff2e88"><b>💡 Hint</b></font></summary>

`for k in range(1, n + 1): total *= k`. Python integers never overflow, so this works for any `n`.

</details>

In [ ]:
# Exercise 1.2: running this cell checks your answer
def num_orders(n: int) -> int:
    ...  # your code here



<font size="4"><b>Twenty stops</b></font>

Give the courier 20 stops. How many digits does the number of orders have?

<details><summary><font color="#ff2e88"><b>💡 Hint</b></font></summary>

`str(x)` writes a whole number out in full, and `len` counts its characters.

</details>

In [ ]:
# Exercise 1.3: running this cell checks your answer
digits_20 = ...  # your answer



<font size="4"><b>Seventy seven years</b></font>

A computer checks a billion ($10^9$) tours every second. How many years does it need to check all $20!$ orders? Use 365.25 days per year.

<details><summary><font color="#ff2e88"><b>💡 Hint</b></font></summary>

Divide $20!$ by $10^9$ to get seconds. Then divide by the seconds in a year: $3600 \times 24 \times 365.25$.

</details>

In [ ]:
# Exercise 1.4: running this cell checks your answer
years_20 = ...  # your answer



<font size="4"><b>The length of one tour</b></font>

Write `tour_length(order, D)`. `order` is a tuple of stops such as `(3, 1, 2, 5, 4)`. The tour starts at the depot 0, visits the stops in that order, and comes back to 0. Return the total distance.

<details><summary><font color="#ff2e88"><b>💡 Hint</b></font></summary>

Build the full list `[0, *order, 0]`, then add `D[a][b]` for every neighbouring pair. `zip(stops, stops[1:])` gives the pairs.

</details>

In [ ]:
# Exercise 1.5: running this cell checks your answer
def tour_length(order: tuple[int, ...], D: np.ndarray) -> float:
    ...  # your code here



<font size="4"><b>Check every tour</b></font>

Write `best_tour(D)` that tries every order of the stops with `itertools.permutations` and returns `(order, length)` for the shortest. The stops are `1` to `len(D) - 1`.

<details><summary><font color="#ff2e88"><b>💡 Hint</b></font></summary>

`itertools.permutations(range(1, len(D)))` gives every order. Use `min(..., key=...)` with your `tour_length`.

</details>

In [ ]:
# Exercise 1.6: running this cell checks your answer
def best_tour(D: np.ndarray) -> tuple[tuple[int, ...], float]:
    ...  # your code here



<font size="4"><b>One more stop</b></font>

Without computing either number in full: how many times more orders does a 21-stop courier have than a 20-stop courier?

<details><summary><font color="#ff2e88"><b>💡 Hint</b></font></summary>

$21! = 21 \times 20!$.

</details>

In [ ]:
# Exercise 1.7: running this cell checks your answer
ratio_21 = ...  # your answer



<font size="4"><b>The fast program&#x27;s work</b></font>

The program from the video does about $n^2 \times 2^n$ steps for $n$ stops. For 20 stops, how many steps is that, and how many seconds at $10^9$ steps per second? Give a tuple `(steps, seconds)`.

<details><summary><font color="#ff2e88"><b>💡 Hint</b></font></summary>

$20^2 \times 2^{20}$, then divide by $10^9$.

</details>

In [ ]:
# Exercise 1.8: running this cell checks your answer
fast_steps, fast_seconds = ..., ...  # your answer



## <font color="#7b2ff7">The recipe: modeling, inference and learning</font>

An **agent** is a program that observes, decides and acts. Then the world changes and it observes again. A spam filter, a route planner and a chess program are all agents. Every agent has a fixed amount of time and memory, and sees only part of the world.

Every method in this course is one of three steps.

* **Modeling**: write the real problem as a precise mathematical object. For directions across a city, the model is a **graph**: each intersection is a **node**, each road is an **edge**, and each edge carries a driving time in minutes.
* **Inference**: answer a question using the model. What is the fastest route from home to the airport?
* **Learning**: fill in the model's numbers from data. Each driving time is the average of past trips along that road.

The steps are kept separate on purpose. If a road time changes, the route algorithm still works without a single change to its code.

Below is the city from the video: home `H`, three intersections `P`, `Q`, `R`, and the airport `Air`.

In [ ]:
type Edges = dict[tuple[str, str], float]

CITY: Edges = {("H", "P"): 4, ("H", "Q"): 6, ("P", "Q"): 3, ("P", "R"): 7,
               ("Q", "R"): 2, ("R", "Air"): 5, ("Q", "Air"): 9}
CITY_POS = {"H": (0, 1), "P": (1, 2), "Q": (1.4, 0.3), "R": (2.5, 1.4), "Air": (3.6, 0.6)}
TRIPS_HQ = [5.5, 6.5, 6.0]

draw_graph(CITY, CITY_POS, title="The city: driving times in minutes")



<font size="4"><b>An adjacency list</b></font>

Inference code wants to ask "which roads leave this node?". Write `build_adjacency(edges)` that returns a dictionary from each node to a list of `(neighbour, cost)` pairs. Roads go both ways, so each edge appears in two lists.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Loop over `edges.items()`. For each `(u, v), c`, append `(v, c)` to the list of `u` and `(u, c)` to the list of `v`. `dict.setdefault(u, [])` creates a list the first time.

</details>

In [ ]:
# Exercise 2.1: running this cell checks your answer
def build_adjacency(edges: Edges) -> dict[str, list[tuple[str, float]]]:
    ...  # your code here



<font size="4"><b>The cost of a route</b></font>

Write `route_cost(route, edges)` that adds up the driving times along a list of nodes such as `["H", "Q", "R", "Air"]`. A road may be stored as `(u, v)` or as `(v, u)`.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

For each neighbouring pair, look up `edges.get((u, v))`, and if that is `None`, `edges[(v, u)]`.

</details>

In [ ]:
# Exercise 2.2: running this cell checks your answer
def route_cost(route: list[str], edges: Edges) -> float:
    ...  # your code here



<font size="4"><b>Home to the airport</b></font>

Use your `route_cost` to find the driving time of the route H, Q, R, Air.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Pass the list of four nodes and `CITY`.

</details>

In [ ]:
# Exercise 2.3: running this cell checks your answer
hqr = ...  # your answer



<font size="4"><b>Every route</b></font>

Write `all_routes(adj, start, goal)`. It returns every route from `start` to `goal` that never visits a node twice, as a list of `(route, cost)` pairs, where `route` is a tuple of nodes. Sort the list by cost, cheapest first.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Write a helper `go(node, path, cost)` that calls itself for each neighbour not already in `path`. When `node == goal`, append `(tuple(path), cost)` to the output list. Sort at the end with `key=lambda r: r[1]`.

</details>

In [ ]:
# Exercise 2.4: running this cell checks your answer
def all_routes(adj: dict[str, list[tuple[str, float]]], start: str, goal: str) -> list[tuple[tuple[str, ...], float]]:
    ...  # your code here



<font size="4"><b>The winner and the runner-up</b></font>

From your routes, give the fastest route and how many minutes it beats the second fastest by.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

The list is sorted: element 0 is the best, element 1 the runner-up. Each element is `(route, cost)`.

</details>

In [ ]:
# Exercise 2.5: running this cell checks your answer
best_city, gap_city = ..., ...  # your answer



<font size="4"><b>Learn a road time</b></font>

Nobody types in every road time. The time for the road H to Q is the average of the past trips in `TRIPS_HQ`. Compute it.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

`np.mean` or `sum(...) / len(...)`.

</details>

In [ ]:
# Exercise 2.6: running this cell checks your answer
hq_learned = ...  # your answer



<font size="4"><b>New data, same inference code</b></font>

A new trip on H to Q takes 11 minutes because of roadworks. Learn the new average, put it into the model, and run your `all_routes` again on the changed model. What is the new fastest route?

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

`{**CITY, ("H", "Q"): hq_new}` is a copy of the model with one number changed. The inference code is the same as before.

</details>

In [ ]:
# Exercise 2.7: running this cell checks your answer
hq_new = ...  # your answer
new_best = ...  # your answer



<font size="4"><b>The phone keyboard</b></font>

A phone keyboard suggests the next word as you type. Label each part with `"modeling"`, `"inference"` or `"learning"`.

* `score_choice`: the choice to score each possible next word from the words typed so far.
* `pick_now`: picking the highest scoring word on every keystroke.
* `scores_from_text`: setting the scores from a large collection of text.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

What is chosen once? What runs on every keystroke? What comes from data?

</details>

In [ ]:
# Exercise 2.8: running this cell checks your answer
score_choice = ...  # your answer
pick_now = ...  # your answer
scores_from_text = ...  # your answer



## <font color="#00a383">Reflex models</font>

The simplest kind of model is a **reflex model**. It takes an input and gives an answer in one fixed computation, with no planning.

A reflex spam filter first turns an email into numbers called **features**: measurements of the input that a program can compute. Here there are three: whether the email contains the word "free" (1 or 0), the number of links, and whether the sender is a known contact (1 or 0). For the email in the video, the features are $\phi = (1, 3, 0)$. The Greek letter $\phi$ (phi) is the usual name for a feature list.

Each feature has a **weight**: a number that says how hard the feature pushes toward spam. Here the weights are $w = (2, 0.5, -3)$. A **bias** $b = -1.5$ sets the score an email would get if every feature were zero.

The **score** multiplies each feature by its weight, adds the products, and adds the bias. A positive score means spam and a negative score means not spam.

To turn a score into a probability, the **sigmoid** function $\sigma(s) = 1 / (1 + e^{-s})$ squashes any number into the range 0 to 1.

In [ ]:
PHI = [1, 3, 0]           # free, links, known contact
W = [2.0, 0.5, -3.0]
B = -1.5

s = np.linspace(-6, 6, 241)
fig, ax = plt.subplots()
ax.plot(s, _sig(s), color=PALETTE["cyan"])
ax.axvline(0, color=PALETTE["ink"], lw=1, ls="--")
ax.scatter([2], [_sig(2)], s=90, color=PALETTE["pink"], zorder=3)
ax.annotate("score 2", (2, _sig(2)), (2.6, 0.62), color=PALETTE["pink"])
ax.set_xlabel("score"); ax.set_ylabel("probability of spam")
ax.set_title("The sigmoid squashes any score into 0 to 1")
plt.show()



<font size="4"><b>The score by hand</b></font>

Compute the score of the email: each feature times its weight, added up, plus the bias. Write the arithmetic out with the numbers.

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

$2 \times 1 + 0.5 \times 3 + (-3) \times 0 + (-1.5)$.

</details>

In [ ]:
# Exercise 3.1: running this cell checks your answer
score_hand = ...  # your answer



<font size="4"><b>A score function</b></font>

Write `score(phi, w, b)` with a plain Python loop over the features. No NumPy yet.

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

Start a total at `b`, then add `w[i] * phi[i]` for every `i`.

</details>

In [ ]:
# Exercise 3.2: running this cell checks your answer
def score(phi: list[float], w: list[float], b: float) -> float:
    ...  # your code here



<font size="4"><b>The verdict</b></font>

Write `verdict(s)` that returns `"spam"` for a positive score and `"not spam"` otherwise. A `match` statement with a guard (`case _ if s > 0:`) reads well here, and an `if` works too.

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

The dividing line is 0.

</details>

In [ ]:
# Exercise 3.3: running this cell checks your answer
def verdict(s: float) -> str:
    ...  # your code here



<font size="4"><b>The sigmoid</b></font>

Write `sigmoid(s)` for a single number, with `math.exp`.

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

$\sigma(s) = 1 / (1 + e^{-s})$. Mind the minus sign.

</details>

In [ ]:
# Exercise 3.4: running this cell checks your answer
def sigmoid(s: float) -> float:
    return ...  # your code here



<font size="4"><b>From a known contact</b></font>

The same email now comes from a known contact, so the third feature changes from 0 to 1. Give the new score and its sigmoid as a tuple.

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

The known contact adds $-3 \times 1$ to the old score of 2.

</details>

In [ ]:
# Exercise 3.5: running this cell checks your answer
contact_score, contact_prob = ..., ...  # your answer



<font size="4"><b>The feature that matters most</b></font>

Reading the weights tells you what the model has learned. Which feature has the largest weight in size (ignoring the sign)? Answer with one of `"free"`, `"links"`, `"known contact"`.

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

Compare $|2|$, $|0.5|$ and $|-3|$.

</details>

In [ ]:
# Exercise 3.6: running this cell checks your answer
strongest = ...  # your answer

With two features you can draw the whole model. A second filter uses the number of links $a$ and the number of exclamation marks $m$:
$$\text{score} = 0.5\,a + 0.25\,m - 2.$$
The score is zero along a straight line called the **decision boundary**. Emails on one side are spam and emails on the other side are not.

In [ ]:
EMAILS = {"e1": (1, 2), "e2": (6, 1), "e3": (2, 6), "e4": (3, 1), "e5": (5, 5)}   # (links, marks)

fig, ax = plt.subplots(figsize=(5.2, 4.2))
for name, (a, m) in EMAILS.items():
    ax.scatter([a], [m], s=160, color=PALETTE["violet"], zorder=3)
    ax.annotate(name, (a, m), (a + 0.15, m + 0.25))
ax.set_xlim(0, 7); ax.set_ylim(0, 8)
ax.set_xlabel("links"); ax.set_ylabel("exclamation marks")
ax.set_title("Five emails, before the boundary is drawn")
plt.show()



<font size="4"><b>Which emails are spam?</b></font>

Score each email in `EMAILS` with the second filter and return the sorted list of names that come out as spam.

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

Loop over `EMAILS.items()`, compute `0.5 * a + 0.25 * m - 2`, and keep the names with a positive score.

</details>

In [ ]:
# Exercise 3.7: running this cell checks your answer
spam_names = ...  # your answer



<font size="4"><b>The boundary line</b></font>

Write `boundary_marks(a)`: the number of marks $m$ that puts an email with $a$ links exactly on the boundary. Solve $0.5\,a + 0.25\,m - 2 = 0$ for $m$. A plot appears when the check passes.

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

Move the other terms across and divide by 0.25.

</details>

In [ ]:
# Exercise 3.8: running this cell checks your answer
def boundary_marks(a: float) -> float:
    return ...  # your code here



<font size="4"><b>The surest verdict</b></font>

The farther an email sits from the line, the bigger its score in size, and the surer the model is. Which email gets the surest verdict?

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

Compare the absolute values of the five scores.

</details>

In [ ]:
# Exercise 3.9: running this cell checks your answer
surest = ...  # your answer



<font size="4"><b>A sigmoid that never overflows</b></font>

`math.exp(1000)` raises `OverflowError`, so your `sigmoid(-1000.0)` crashes. Write `stable_sigmoid(s)` that works for any score. For `s >= 0` use the usual formula. For `s < 0` use the equal form $e^{s} / (1 + e^{s})$, where the exponent is negative and can't overflow.

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

Two branches with an `if`. Both formulas give the same number wherever both can be computed.

</details>

In [ ]:
# Exercise 3.10: running this cell checks your answer
def stable_sigmoid(s: float) -> float:
    ...  # your code here



## <font color="#ff5f2e">Vectors, matrices and shapes</font>

A **vector** is a list of numbers in a fixed order. The email's features form one vector and the weights form another, in the same order: free, links, known contact. Entry 1 of the weights belongs to entry 1 of the features.

Multiply two vectors entry by entry and add the products: that is the **dot product**, written $w \cdot \phi$. It is the first part of the score. In NumPy it is `w @ phi`.

Stack several emails as rows and you get a **matrix**: a grid of numbers with one row per email and one column per feature. Its **shape** is (rows, columns). Multiplying a (4, 3) matrix by a vector of 3 takes one dot product per row and gives 4 numbers.

Adding the single bias to all 4 scores is called **broadcasting**: the library stretches the single number across the rows, so you never write a loop.

In [ ]:
w_vec = np.array(W)
phi_vec = np.array(PHI, dtype=float)
X4 = np.array([[1, 3, 0],
               [0, 1, 1],
               [1, 6, 0],
               [0, 2, 0]], dtype=float)

fig, ax = plt.subplots(figsize=(4.4, 3.6))
ax.imshow(X4, cmap="cool", aspect="auto")
for i, j in itertools.product(range(4), range(3)):
    ax.text(j, i, f"{X4[i, j]:g}", ha="center", va="center", color="#0c0922", weight="bold")
ax.set_xticks(range(3), ["free", "links", "contact"]); ax.set_yticks(range(4), [f"email {i + 1}" for i in range(4)])
ax.grid(False); ax.set_title("Four emails as a (4, 3) matrix")
plt.show()



<font size="4"><b>The dot product</b></font>

Compute $w \cdot \phi$ with `@`, using `w_vec` and `phi_vec`.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

`a @ b` for two vectors of the same length.

</details>

In [ ]:
# Exercise 4.1: running this cell checks your answer
dot = ...  # your answer



<font size="4"><b>The shape of the batch</b></font>

What is `X4.shape`? Type the tuple yourself, then check it in a scratch cell.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

Rows first, then columns.

</details>

In [ ]:
# Exercise 4.2: running this cell checks your answer
x4_shape = ...  # your answer



<font size="4"><b>Four scores at once</b></font>

Score all four emails in one line: matrix times weight vector, plus the bias.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

`X4 @ w_vec + B`. The `+ B` is broadcasting.

</details>

In [ ]:
# Exercise 4.3: running this cell checks your answer
scores4 = ...  # your answer



<font size="4"><b>Flag the spam</b></font>

Make a boolean array that is `True` for each email with a positive score.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

Comparing an array with a number, `scores4 > 0`, compares every entry.

</details>

In [ ]:
# Exercise 4.4: running this cell checks your answer
flags4 = ...  # your answer



<font size="4"><b>Count the multiply-adds</b></font>

How many multiply-adds does `X4 @ w_vec` do? Use an attribute of the array.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

One per entry of the matrix: `X4.size`.

</details>

In [ ]:
# Exercise 4.5: running this cell checks your answer
mult_adds = ...  # your answer



<font size="4"><b>Predict a shape</b></font>

Predict the shape of `A @ v` when `A` has shape (32, 3) and `v` has shape (3,). Write it as a tuple.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

The inner sizes, 3 and 3, match and disappear. The outer size survives.

</details>

In [ ]:
# Exercise 4.6: running this cell checks your answer
shape_pred = ...  # your answer



<font size="4"><b>Catch the shape error</b></font>

Write `try_matmul(A, v)` that returns the shape of `A @ v`, or the string `"error"` if NumPy raises a `ValueError` because the inner sizes disagree.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

Put `return (A @ v).shape` inside `try:` and `return "error"` inside `except ValueError:`.

</details>

In [ ]:
# Exercise 4.7: running this cell checks your answer
def try_matmul(A: np.ndarray, v: np.ndarray) -> tuple[int, ...] | str:
    ...  # your code here



<font size="4"><b>A batch stored on its side</b></font>

`sideways` holds 32 emails stored as columns, shape (3, 32). Turning a matrix over so rows become columns is called a **transpose**, written `.T`. Score all 32 emails with `w_vec` and `B`.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

`sideways.T` has shape (32, 3). Then multiply by the weights and add the bias.

</details>

In [ ]:
# Exercise 4.8: running this cell checks your answer
sideways = np.random.default_rng(3).integers(0, 4, size=(3, 32)).astype(float)
scores32 = ...  # your answer



<font size="4"><b>A broadcasting trap</b></font>

`s4` below has shape (4,). Predict the shape of `s4[:, None] + s4`, then check it in a scratch cell. `[:, None]` adds an axis of length 1.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

A (4, 1) block plus a (4,) vector: NumPy stretches each length-1 axis to match the other side.

</details>

In [ ]:
# Exercise 4.9: running this cell checks your answer
s4 = np.array([2.0, -4.0, 3.5, -0.5])
trap_shape = ...  # your answer



<font size="4"><b>The same scores in PyTorch</b></font>

PyTorch stores numbers in a **tensor**, its version of a NumPy array. Make float64 tensors from `X4` and `w_vec` and compute the four scores with `@` and `+ B`.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

`torch.tensor(X4, dtype=torch.float64)`. The `@` and broadcasting rules are the same as in NumPy.

</details>

In [ ]:
# Exercise 4.10: running this cell checks your answer
scores_torch = ...  # your answer



## <font color="#d0309f">State-based models</font>

A reflex model can't plan. When the answer is a sequence of decisions, and each choice changes the next one, we need a model that keeps track of where we are.

In a **state-based model**, a **state** is everything you need to know about where you are right now. An **action** moves you from one state to another, and each action has a **cost**. A **plan** is a sequence of actions from the start to the goal, and its cost is the sum along the way.

Below is the courier's small road network from the video. The state is the current intersection, and each road is an action that costs its minutes. Answering questions on a state-based model is called **search**.

In [ ]:
ROADS: Edges = {("S", "A"): 1, ("S", "B"): 4, ("A", "B"): 2, ("A", "C"): 5,
                ("B", "C"): 1, ("C", "G"): 3, ("B", "G"): 6}
ROADS_POS = {"S": (0, 1), "A": (1, 2), "B": (1.2, 0.2), "C": (2.4, 1.5), "G": (3.4, 0.5)}
draw_graph(ROADS, ROADS_POS, title="From S to G: road costs in minutes", node_colour="#3dffb4")



<font size="4"><b>Actions out of S</b></font>

Use your `build_adjacency` on `ROADS` and give the sorted list of `(neighbour, cost)` actions out of S.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

`sorted(build_adjacency(ROADS)["S"])`.

</details>

In [ ]:
# Exercise 5.1: running this cell checks your answer
actions_S = ...  # your answer



<font size="4"><b>Count the plans</b></font>

How many paths from S to G never revisit an intersection? Use your `all_routes`.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

`len(all_routes(build_adjacency(ROADS), "S", "G"))`.

</details>

In [ ]:
# Exercise 5.2: running this cell checks your answer
n_paths = ...  # your answer



<font size="4"><b>The cheapest plan</b></font>

Give the cheapest path as a tuple of nodes, and its cost.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

The first element of the sorted list.

</details>

In [ ]:
# Exercise 5.3: running this cell checks your answer
best_path, best_cost = ..., ...  # your answer



<font size="4"><b>The range of costs</b></font>

Give `(cheapest, dearest)` over all the paths.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

The costs are the second item of each pair. `min` and `max` of them.

</details>

In [ ]:
# Exercise 5.4: running this cell checks your answer
cost_range = ...  # your answer



<font size="4"><b>Cost so far</b></font>

Search tracks the cost so far. Give the running total after each road along S, A, B, C, G as a list: after the first road, after the second, and so on.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

`itertools.accumulate` or `np.cumsum` over the four road costs 1, 2, 1, 3.

</details>

In [ ]:
# Exercise 5.5: running this cell checks your answer
cost_so_far = ...  # your answer



<font size="4"><b>A road closes</b></font>

The road from B to C closes. Remove it from the model, find the new best cost, and count how many paths reach it.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

`{k: v for k, v in ROADS.items() if k != ("B", "C")}` copies the model without one road.

</details>

In [ ]:
# Exercise 5.6: running this cell checks your answer
closed = {k: v for k, v in ROADS.items() if k != ("B", "C")}
closed_cost = ...  # your answer
n_tied = ...  # your answer



<font size="4"><b>Order matters</b></font>

Swap two decisions: drive S, B, A, C, G instead of S, A, B, C, G. Use your `route_cost` to find how many minutes the swap costs.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

The cost of the swapped route minus 7.

</details>

In [ ]:
# Exercise 5.7: running this cell checks your answer
swap_penalty = ...  # your answer



<font size="4"><b>The tree of futures</b></font>

Unroll the choices into a **tree**: the start at the top, one branch per action, and let the courier double back if she wants. Write `tree_level_sizes(adj, start, depth)` that returns how many branches end at each level, from level 0 (just the start) to level `depth`.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

Keep a list of paths. Level 0 is `[[start]]`. Each next level is `[p + [v] for p in level for v, _ in adj[p[-1]]]`. Record `len(level)` each time.

</details>

In [ ]:
# Exercise 5.8: running this cell checks your answer
def tree_level_sizes(adj: dict[str, list[tuple[str, float]]], start: str, depth: int) -> list[int]:
    ...  # your code here



<font size="4"><b>The same places again</b></font>

Look at the 19 branches at level 3 and where each one ends. How many end at B, and how many different places appear? Give a tuple. A plot appears when the check passes.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

Build the level-3 paths as in the last exercise, then `Counter(p[-1] for p in paths)`.

</details>

In [ ]:
# Exercise 5.9: running this cell checks your answer
ends_at_B, n_places = ..., ...  # your answer



## <font color="#0077ff">Variable-based models</font>

Some problems have no sequence at all. This map has 4 regions, A, B, C and D. We want to colour each region red, green or blue so that no two neighbours share a colour.

In a **variable-based model**, the unknowns are **variables**, and each variable takes a **value**. Here there are 4 variables, one per region, each with 3 possible colours. The rules between variables are **constraints**: each shared border says its two regions must differ.

An **assignment** gives a value to some of the variables. A **complete assignment** gives every region a colour, and it is **valid** when every constraint holds. Only the final assignment is judged, so you may colour the regions in any order.

In [ ]:
REGIONS = ["A", "B", "C", "D"]
MAP_EDGES = [("A", "B"), ("A", "C"), ("B", "C"), ("C", "D"), ("B", "D")]
COLOURS = ["red", "green", "blue"]

MAP_POS = {"A": (0, 1), "B": (1, 2), "C": (1, 0), "D": (2, 1)}
fig, ax = plt.subplots(figsize=(4.6, 3.4))
for u, v in MAP_EDGES:
    ax.plot(*zip(MAP_POS[u], MAP_POS[v]), color="#5a4fa0", zorder=1)
for n, (x, y) in MAP_POS.items():
    ax.scatter([x], [y], s=900, color=PALETTE["violet"], zorder=2)
    ax.text(x, y, n, ha="center", va="center", color="#0c0922", weight="bold", zorder=3)
ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
ax.set_title("Regions as nodes, shared borders as lines")
plt.show()



<font size="4"><b>All assignments</b></font>

How many complete assignments are there, valid or not?

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

3 choices for each of 4 regions.

</details>

In [ ]:
# Exercise 6.1: running this cell checks your answer
n_colourings = ...  # your answer



<font size="4"><b>Is a colouring valid?</b></font>

Write `is_valid(colouring, edges)`. `colouring` is a dictionary such as `{"A": "red", "B": "green", ...}`. Return `True` when no edge joins two regions of the same colour.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

`all(colouring[u] != colouring[v] for u, v in edges)`.

</details>

In [ ]:
# Exercise 6.2: running this cell checks your answer
def is_valid(colouring: dict[str, str], edges: list[tuple[str, str]]) -> bool:
    return ...  # your code here



<font size="4"><b>The valid colourings</b></font>

Build the list of every valid colouring, as dictionaries. `itertools.product(COLOURS, repeat=4)` gives every complete assignment as a tuple.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

`dict(zip(REGIONS, combo))` turns a tuple into a dictionary. Keep the ones your `is_valid` accepts.

</details>

In [ ]:
# Exercise 6.3: running this cell checks your answer
valid = ...  # your answer



<font size="4"><b>Thrown out early</b></font>

Colour A and B both red, and the border between them is broken whatever C and D do. How many of the 81 complete assignments does that one broken constraint rule out?

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

A and B are fixed. C and D are free: 3 choices each.

</details>

In [ ]:
# Exercise 6.4: running this cell checks your answer
ruled_out = ...  # your answer



<font size="4"><b>One region at a time</b></font>

Colour A, then B, then C, then D, and count how many colours are still allowed at each step. Give the list of four counts.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

A is free. B touches A. C touches A and B. D touches B and C, which already differ.

</details>

In [ ]:
# Exercise 6.5: running this cell checks your answer
chain = ...  # your answer



<font size="4"><b>A general counter</b></font>

Write `count_valid(regions, edges, colours)` that counts the valid colourings of any map with any list of colours.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

The same product loop as before, with `repeat=len(regions)` and the given edges.

</details>

In [ ]:
# Exercise 6.6: running this cell checks your answer
def count_valid(regions: list[str], edges: list[tuple[str, str]], colours: list[str]) -> int:
    ...  # your code here



<font size="4"><b>A ring of four</b></font>

Four regions sit in a ring: A borders B, B borders C, C borders D, and D borders A. How many valid colourings with 3 colours?

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

Use your `count_valid` with the four ring edges.

</details>

In [ ]:
# Exercise 6.7: running this cell checks your answer
ring = ...  # your answer



<font size="4"><b>The swap test</b></font>

Would swapping the order of two decisions change the answer? If yes, the problem is state-based; if no, it is variable-based. Label each problem `"state"` or `"variable"`.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

Filling one Sudoku cell before another makes no difference to which grids are valid. Turning left before right gives a different route.

</details>

In [ ]:
# Exercise 6.8: running this cell checks your answer
sudoku, directions, seating, chess_game = ..., ..., ..., ...  # your answer



## <font color="#7b2ff7">Logic-based models</font>

Suppose we know two rules. If it rains, the ground is wet. If the ground is wet, the road is slippery.

Give each statement a short name: Rain, Wet and Slippery. Each is either true or false. A rule "if P then Q" is written $P \Rightarrow Q$ and is called an **implication**. It is broken in exactly one case: P is true and Q is false.

Three true-or-false statements give 8 possible **worlds**, one per row of a **truth table**. A world where every rule holds is called a **model** of the rules. When a conclusion is true in every model that's still possible, the facts **entail** the conclusion.

Below, a world is a tuple `(rain, wet, slippery)` of booleans.

In [ ]:
fig, ax = plt.subplots(figsize=(5.4, 4.0))
rows8 = np.array(list(itertools.product([0, 1], repeat=3)))
ax.imshow(rows8, cmap="cool", aspect="auto")
for i, j in itertools.product(range(8), range(3)):
    ax.text(j, i, "T" if rows8[i, j] else "F", ha="center", va="center", color="#0c0922", weight="bold")
ax.set_xticks(range(3), ["Rain", "Wet", "Slippery"]); ax.set_yticks(range(8), [f"world {i}" for i in range(8)])
ax.grid(False); ax.set_title("8 possible worlds")
plt.show()



<font size="4"><b>Implication</b></font>

Write `implies(p, q)`: `False` only when `p` is true and `q` is false.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

`(not p) or q`.

</details>

In [ ]:
# Exercise 7.1: running this cell checks your answer
def implies(p: bool, q: bool) -> bool:
    return ...  # your code here



<font size="4"><b>Every world</b></font>

Build the list of all 8 worlds as tuples of booleans, in the order `itertools.product` gives.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

`itertools.product([False, True], repeat=3)`.

</details>

In [ ]:
# Exercise 7.2: running this cell checks your answer
worlds = ...  # your answer



<font size="4"><b>Models of the rules</b></font>

Keep the worlds where both rules hold: Rain implies Wet, and Wet implies Slippery.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

A list comprehension over `worlds` with `implies(r, w) and implies(w, s)`.

</details>

In [ ]:
# Exercise 7.3: running this cell checks your answer
rule_models = ...  # your answer



<font size="4"><b>Now it rains</b></font>

Add the fact that it is raining. Which worlds remain?

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Keep the models where the first entry is true.

</details>

In [ ]:
# Exercise 7.4: running this cell checks your answer
with_rain = ...  # your answer



<font size="4"><b>Entailment</b></font>

Write `entails(models, i)`: `True` when statement number `i` is true in every world in `models`.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

`all(m[i] for m in models)`.

</details>

In [ ]:
# Exercise 7.5: running this cell checks your answer
def entails(models: list[tuple[bool, ...]], i: int) -> bool:
    return ...  # your code here



<font size="4"><b>A slippery road</b></font>

Start again from the two rules. The only extra fact is that the road is slippery. How many worlds remain, and does it follow that it rained? Give `(count, rain_follows)`.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Keep the models where the third entry is true, then use your `entails` on statement 0.

</details>

In [ ]:
# Exercise 7.6: running this cell checks your answer
slip_count, rain_follows = ..., ...  # your answer



<font size="4"><b>A dry ground</b></font>

This time the extra fact is that the ground is not wet. Is "no rain" entailed? Answer `True` or `False`.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Keep the models with `wet == False` and look at the rain entry in each.

</details>

In [ ]:
# Exercise 7.7: running this cell checks your answer
no_rain = ...  # your answer



<font size="4"><b>Chaining the rules</b></font>

You can also read the proof off the rules: Rain gives Wet, and Wet gives Slippery. Write `forward_chain(facts, rules)`. `facts` is a set of names, and `rules` is a list of `(if_name, then_name)` pairs. Keep applying rules until nothing new appears, then return the set.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Copy the set, then loop: for each rule whose first name is known and second name is not, add the second name. Stop when a full pass adds nothing.

</details>

In [ ]:
# Exercise 7.8: running this cell checks your answer
def forward_chain(facts: set[str], rules: list[tuple[str, str]]) -> set[str]:
    ...  # your code here



<font size="4"><b>Rows double</b></font>

Each statement you add doubles the number of rows. What is the smallest number of statements whose truth table has more than a billion ($10^9$) rows?

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Find the smallest `n` with `2 ** n > 1e9`.

</details>

In [ ]:
# Exercise 7.9: running this cell checks your answer
n_billion = ...  # your answer



## <font color="#e52e71">The ladder of models</font>

The four kinds of model form a **ladder**: reflex at the bottom, then state-based, then variable-based, and logic-based at the top. Each rung can describe more problems than the one below it.

The price is computation:

* a reflex model does one multiply-add per feature;
* a state-based model searches a tree of possible futures;
* a variable-based model faces every possible assignment;
* a logic-based model, checked row by row, faces every possible world.

The plot below draws the size of a search tree on a **log scale**, where each tick up is 10 times more.

In [ ]:
depths = np.arange(0, 11)
fig, ax = plt.subplots()
ax.semilogy(depths, 3.0 ** depths, "o-", color=PALETTE["orange"], label="3 actions per step")
ax.semilogy(depths, 2.0 ** depths, "o-", color=PALETTE["cyan"], label="2 actions per step")
ax.set_xlabel("steps"); ax.set_ylabel("leaves (log scale)")
ax.set_title("A tree of futures grows by a factor at every step")
ax.legend()
plt.show()



<font size="4"><b>Leaves of a tree</b></font>

A state-based model has 3 actions per step. How many leaves does its tree have after 10 steps?

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

Each step multiplies the branches by 3.

</details>

In [ ]:
# Exercise 8.1: running this cell checks your answer
leaves = ...  # your answer



<font size="4"><b>Every node of that tree</b></font>

Counting every level from 0 to 10, how many nodes does that tree have?

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

Add $3^k$ for $k = 0, 1, \dots, 10$.

</details>

In [ ]:
# Exercise 8.2: running this cell checks your answer
tree_nodes = ...  # your answer



<font size="4"><b>Twenty regions</b></font>

A variable-based model colours 20 regions with 3 colours. How many complete assignments?

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

3 choices, 20 times.

</details>

In [ ]:
# Exercise 8.3: running this cell checks your answer
colourings_20 = ...  # your answer



<font size="4"><b>Thirty statements</b></font>

A logic-based model has 30 true-or-false statements. How many rows does its truth table have?

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

2 values, 30 times.

</details>

In [ ]:
# Exercise 8.4: running this cell checks your answer
rows_30 = ...  # your answer



<font size="4"><b>Draw the ladder</b></font>

Make the base-10 logs of the four costs: 3 multiply-adds for the reflex model, then `leaves`, `colourings_20` and `rows_30`. Return a NumPy array of 4 numbers. A plot appears when the check passes.

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

`np.log10(np.array([3, leaves, colourings_20, rows_30], dtype=float))`.

</details>

In [ ]:
# Exercise 8.5: running this cell checks your answer
log_costs = ...  # your answer



<font size="4"><b>How long does it take?</b></font>

At $10^9$ checks per second, how many seconds do the 20-region colourings take? And how many years for 40 regions? Give a tuple, with 365.25 days per year.

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

$3^{20} / 10^9$ seconds. For 40 regions, $3^{40} / 10^9$ seconds, divided by $3600 \times 24 \times 365.25$.

</details>

In [ ]:
# Exercise 8.6: running this cell checks your answer
secs_20, years_40 = ..., ...  # your answer



<font size="4"><b>One more of each</b></font>

By what factor does the work grow when you add one more step to the 3-action tree, one more region to the 3-colour map, and one more statement to the truth table? Give a tuple of three numbers.

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

Each step has 3 actions, each region 3 colours, each statement 2 values.

</details>

In [ ]:
# Exercise 8.7: running this cell checks your answer
growth = ...  # your answer



<font size="4"><b>The lowest rung that fits</b></font>

In practice you choose the lowest rung that can still describe your problem. Label each with `"reflex"`, `"state"`, `"variable"` or `"logic"`.

* `photo`: decide whether a photo shows a cat.
* `maze`: plan moves through a maze.
* `wedding`: seat 40 guests at tables so that certain pairs are apart.
* `rulebook`: check whether a claim follows from a list of eligibility rules.

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

One fixed computation; a sequence where order matters; a final assignment; a proof from stated rules.

</details>

In [ ]:
# Exercise 8.8: running this cell checks your answer
photo, maze, wedding, rulebook = ..., ..., ..., ...  # your answer



## <font color="#00897b">Sums and products</font>

Two pieces of notation say exactly what learning does.

The capital Greek letter **sigma**, $\sum$, means add up. $\sum_{i=1}^{n} a_i$ means $a_1 + a_2 + \dots + a_n$: the index $i$ starts at 1 (written below) and stops at $n$ (written on top). A sigma is a loop written on one line.

The reflex score is a sigma: $\text{score} = \sum_{i=1}^{3} w_i \phi_i + b$.

The **loss** that learning makes small is a sigma too. For $n$ examples with inputs $x_i$, true outputs $y_i$ and predictions $\hat{y}_i$:
$$L = \frac{1}{n} \sum_{i=1}^{n} (\hat{y}_i - y_i)^2.$$
That is the average squared error.

The capital Greek letter **pi**, $\prod$, means multiply, with the same index and limits.

Products of probabilities shrink fast. The plot shows $0.5^n$, the chance that $n$ fair coin flips all land heads.

In [ ]:
ns = np.arange(1, 1201)
vals = 0.5 ** ns.astype(float)
fig, ax = plt.subplots()
ax.semilogy(ns[vals > 0], vals[vals > 0], color=PALETTE["pink"])
first_zero = int(ns[vals == 0][0])
ax.axvline(first_zero, color=PALETTE["amber"], ls="--", lw=1.5)
ax.text(first_zero - 30, 1e-150, f"0.5^{first_zero} rounds to 0", color=PALETTE["amber"], ha="right")
ax.set_xlabel("coin flips n"); ax.set_ylabel("0.5^n (log scale)")
ax.set_title("A product of probabilities falls off the bottom")
plt.show()



<font size="4"><b>A sigma as a loop</b></font>

Write `sigma(f, lo, hi)` that returns $\sum_{i=\text{lo}}^{\text{hi}} f(i)$, including both ends.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

`range(lo, hi + 1)`: Python's `range` leaves out its end.

</details>

In [ ]:
# Exercise 9.1: running this cell checks your answer
def sigma(f, lo: int, hi: int) -> float:
    return ...  # your code here



<font size="4"><b>Read it, then compute it</b></font>

Compute $\sum_{i=1}^{4} i^2$ with your `sigma`.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

`sigma(lambda i: i ** 2, 1, 4)`.

</details>

In [ ]:
# Exercise 9.2: running this cell checks your answer
sum_sq = ...  # your answer



<font size="4"><b>Where the bias sits</b></font>

Compute the score two ways for the video's email: with the bias outside the sigma, $\sum_i w_i \phi_i + b$, and with it moved inside, $\sum_i (w_i \phi_i + b)$. Give a tuple.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

Inside the brackets, the bias is added once per term: 3 times.

</details>

In [ ]:
# Exercise 9.3: running this cell checks your answer
outside, inside = ..., ...  # your answer

The video evaluates the loss on three examples with inputs 1, 2, 3 and outputs 2, 4, 5, using a line through zero that predicts $\hat{y} = w x$ with a trial weight $w = 1.5$.

In [ ]:
X_LEARN = np.array([1.0, 2.0, 3.0])
Y_LEARN = np.array([2.0, 4.0, 5.0])



<font size="4"><b>The squared errors, term by term</b></font>

For $w = 1.5$, compute the array of squared errors $(w x_i - y_i)^2$, one per example.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

NumPy works on whole arrays: `(1.5 * X_LEARN - Y_LEARN) ** 2`.

</details>

In [ ]:
# Exercise 9.4: running this cell checks your answer
sq_errors = ...  # your answer



<font size="4"><b>The loss as a function</b></font>

Write `mse(w, x, y)`: the mean squared error of the line $\hat{y} = w x$ on arrays `x` and `y`. MSE is short for mean squared error.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

`np.mean((w * x - y) ** 2)`.

</details>

In [ ]:
# Exercise 9.5: running this cell checks your answer
def mse(w: float, x: np.ndarray, y: np.ndarray) -> float:
    return ...  # your code here



<font size="4"><b>Empty sums and products</b></font>

What does a sum over no terms give, and a product over no terms? Use `sum([])` and `math.prod([])` and give a tuple.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

Each starts from the value that changes nothing: 0 for adding, 1 for multiplying.

</details>

In [ ]:
# Exercise 9.6: running this cell checks your answer
empty = ...  # your answer



<font size="4"><b>Find the bug</b></font>

The function below is meant to multiply a list of numbers, and it always returns 0. Write a fixed `product_of(values)`.

```python
def broken_product(values):
    total = 0
    for v in values:
        total *= v
    return total
```

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

A product loop has to start from 1.

</details>

In [ ]:
# Exercise 9.7: running this cell checks your answer
def product_of(values: list[float]) -> float:
    ...  # your code here



<font size="4"><b>Where the product hits zero</b></font>

Floats can't hold numbers smaller than about $10^{-324}$. Find the smallest whole number `n` for which `0.5 ** n` comes out as exactly `0.0`.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

Loop `n` upward from 1 until `0.5 ** n == 0.0`.

</details>

In [ ]:
# Exercise 9.8: running this cell checks your answer
n_zero = ...  # your answer



<font size="4"><b>Logs to the rescue</b></font>

The log of a product is the sum of the logs. Compute $\log(0.5^{1100})$ as a sum of 1,100 logs, without ever forming the product.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

`1100 * math.log(0.5)`, or `sum(math.log(0.5) for _ in range(1100))`.

</details>

In [ ]:
# Exercise 9.9: running this cell checks your answer
log_1100 = ...  # your answer



<font size="4"><b>Five coin flips</b></font>

Write the chance that 5 independent fair flips all land heads, first as a product, then as a sum of logs. Give a tuple `(product, log_sum)`.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

`math.prod([0.5] * 5)` and `sum(math.log(0.5) for _ in range(5))`.

</details>

In [ ]:
# Exercise 9.10: running this cell checks your answer
coin_prod, coin_log = ..., ...  # your answer



## <font color="#ff4d6d">Learning a weight</font>

Back to the three examples: input 1 gives 2, input 2 gives 4, input 3 gives 5. The model is a line through zero, $\hat{y} = w x$, and the **weight** $w$ is the only number to choose. Learning means choosing the weight that makes the loss small.

Plotted against $w$, the loss is a bowl. For one weight there is a formula for its lowest point:
$$w^\ast = \frac{\sum_i x_i y_i}{\sum_i x_i^2}.$$

Most models have no such formula, so we walk downhill. The **gradient** is the slope of the loss: how fast the loss changes when $w$ moves a little. For the mean squared error,
$$\frac{dL}{dw} = \frac{1}{n}\sum_{i=1}^{n} 2 (w x_i - y_i)\, x_i.$$
**Gradient descent** steps against the slope, $w \leftarrow w - \eta \frac{dL}{dw}$, where the **step size** $\eta$ (eta) sets how far.

In [ ]:
wgrid = np.linspace(-0.2, 3.4, 300)
bowl = np.array([np.mean((w * X_LEARN - Y_LEARN) ** 2) for w in wgrid])
fig, ax = plt.subplots()
ax.plot(wgrid, bowl, color=PALETTE["cyan"])
ax.scatter([0, 1.5], [np.mean(Y_LEARN ** 2), 0.5], s=80, color=PALETTE["pink"], zorder=3)
ax.annotate("w = 0: loss 15", (0, 15), (0.3, 16), color=PALETTE["pink"])
ax.annotate("w = 1.5: loss 0.5", (1.5, 0.5), (1.6, 4), color=PALETTE["pink"])
ax.set_xlabel("w"); ax.set_ylabel("loss")
ax.set_title("The loss is a bowl in w")
plt.show()



<font size="4"><b>Predictions</b></font>

Write `predict(w, x)` that returns the predictions of the line for an array of inputs.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

The weight times the inputs.

</details>

In [ ]:
# Exercise 10.1: running this cell checks your answer
def predict(w: float, x: np.ndarray) -> np.ndarray:
    return ...  # your code here



<font size="4"><b>Search a grid</b></font>

Evaluate your `mse` at 301 weights from 0 to 3 and return the weight with the smallest loss.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

`ws = np.linspace(0, 3, 301)`, then a list of losses, then `ws[np.argmin(losses)]`.

</details>

In [ ]:
# Exercise 10.2: running this cell checks your answer
ws = np.linspace(0, 3, 301)
w_grid = ...  # your answer



<font size="4"><b>The formula</b></font>

Compute $w^\ast$ with the formula.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

`(X_LEARN @ Y_LEARN) / (X_LEARN @ X_LEARN)`: that is 25 over 14.

</details>

In [ ]:
# Exercise 10.3: running this cell checks your answer
w_star = ...  # your answer



<font size="4"><b>The gradient</b></font>

Write `grad(w, x, y)`: the slope of the mean squared error at `w`.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

`np.mean(2 * (w * x - y) * x)`.

</details>

In [ ]:
# Exercise 10.4: running this cell checks your answer
def grad(w: float, x: np.ndarray, y: np.ndarray) -> float:
    return ...  # your code here



<font size="4"><b>Which example pulls hardest?</b></font>

At $w = 0$, compute the array of gradient terms $2 (w x_i - y_i) x_i$, one per example, and the index of the term largest in size.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

The third example has the biggest input and the biggest miss.

</details>

In [ ]:
# Exercise 10.5: running this cell checks your answer
terms0 = ...  # your answer
pulls_hardest = ...  # your answer



<font size="4"><b>One step by hand</b></font>

Take one gradient descent step from $w = 0$ with step size 0.05. Give the new weight and the new loss.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

$w_1 = 0 - 0.05 \times (-16.67)$. Then use your `mse`.

</details>

In [ ]:
# Exercise 10.6: running this cell checks your answer
w1 = ...  # your answer
loss1 = ...  # your answer



<font size="4"><b>Gradient descent</b></font>

Write `gradient_descent(w0, lr, steps)` that starts at `w0`, takes `steps` steps with step size `lr` on `X_LEARN`, `Y_LEARN`, and returns the list of weights, starting with `w0`. A plot appears when the check passes.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

A `for` loop that appends `w` after each update `w = w - lr * grad(w, X_LEARN, Y_LEARN)`.

</details>

In [ ]:
# Exercise 10.7: running this cell checks your answer
def gradient_descent(w0: float, lr: float, steps: int) -> list[float]:
    ...  # your code here



<font size="4"><b>Too big a step</b></font>

Run 30 steps with step size 0.2 and with 0.25. Label each run `"converges"` or `"diverges"`. Then compute the largest safe step size, $2 / c$, where $c = \frac{1}{n}\sum_i 2 x_i^2$ is how fast the slope changes.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

Look at the last weight of each run. For $c$: `np.mean(2 * X_LEARN ** 2)`.

</details>

In [ ]:
# Exercise 10.8: running this cell checks your answer
at_02, at_025 = ..., ...  # your answer
lr_limit = ...  # your answer



<font size="4"><b>The slope from PyTorch</b></font>

PyTorch computes gradients for you. Make `w0` a float64 tensor with `requires_grad=True`, compute the loss, call `.backward()`, and read `w0.grad`. Give the gradient as a float.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

After `loss0.backward()`, the gradient sits in `w0.grad`, and `.item()` turns it into a float.

</details>

In [ ]:
# Exercise 10.9: running this cell checks your answer
w0 = torch.tensor(0.0, dtype=torch.float64, requires_grad=True)
loss0 = ((w0 * torch.tensor(X_LEARN) - torch.tensor(Y_LEARN)) ** 2).mean()
grad_auto = ...  # call backward, then read the gradient



<font size="4"><b>Fit with an optimizer</b></font>

Write the training loop: 50 steps of `torch.optim.SGD` with step size 0.05. Each step clears the old gradient, computes the loss, calls `backward`, and steps.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

Inside the loop: `opt.zero_grad()`, then `loss = ((w_t * xt - yt) ** 2).mean()`, then `loss.backward()`, then `opt.step()`.

</details>

In [ ]:
# Exercise 10.10: running this cell checks your answer
w_t = torch.zeros((), dtype=torch.float64, requires_grad=True)
opt = torch.optim.SGD([w_t], lr=0.05)
xt, yt = torch.tensor(X_LEARN), torch.tensor(Y_LEARN)
...  # your code here



<font size="4"><b>A prediction</b></font>

With the learned weight $w^\ast$, what does the model predict for an input of 4?

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

The weight times 4.

</details>

In [ ]:
# Exercise 10.11: running this cell checks your answer
pred_4 = ...  # your answer



## <font color="#1e88e5">Named axes</font>

A block of numbers with any number of directions is called a **tensor**, and each direction you can index along is an **axis**. A vector has 1 axis and a matrix has 2.

A shape such as (2, 3, 4) says how long each axis is. It says nothing about what each axis means, and that's where mistakes hide.

Below is the block from the video: 2 sentences (the **batch** axis), 3 words each, 4 features per word. The library **einops** lets you write axes as words: `"batch word feature -> batch feature"` keeps batch and feature and reduces away word. An axis on both sides of the arrow is kept; an axis missing on the right is reduced.

In [ ]:
T = np.array([[[1, 0, 2, 3], [3, 2, 0, 1], [2, 1, 1, 2]],
              [[0, 4, 1, 1], [2, 0, 3, 1], [1, 2, 2, 4]]], dtype=float)

fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.6))
for k, ax in enumerate(axes):
    ax.imshow(T[k], cmap="magma", vmin=0, vmax=4)
    for i, j in itertools.product(range(3), range(4)):
        ax.text(j, i, f"{T[k, i, j]:g}", ha="center", va="center", color="#ece8ff" if T[k, i, j] < 2.5 else "#0c0922")
    ax.set_xticks(range(4), [f"f{j}" for j in range(4)]); ax.set_yticks(range(3), [f"word {i}" for i in range(3)])
    ax.grid(False); ax.set_title(f"sentence {k}")
plt.tight_layout(); plt.show()



<font size="4"><b>The shape and the axes</b></font>

Give `T.shape` and the number of axes `T.ndim` as a tuple `(shape, ndim)`.

<details><summary><font color="#1e88e5"><b>💡 Hint</b></font></summary>

Batch, word, feature.

</details>

In [ ]:
# Exercise 11.1: running this cell checks your answer
t_shape, t_ndim = ..., ...  # your answer



<font size="4"><b>Average over the words</b></font>

Turn each sentence into one row of features by averaging over its words, with NumPy's `.mean(axis=...)`. Axes count from 0.

<details><summary><font color="#1e88e5"><b>💡 Hint</b></font></summary>

Batch is axis 0, word is axis 1, feature is axis 2.

</details>

In [ ]:
# Exercise 11.2: running this cell checks your answer
per_sentence = ...  # your answer



<font size="4"><b>The wrong axis</b></font>

Type `axis=2` by mistake and you average over the features. Predict the shape of `T.mean(axis=2)`.

<details><summary><font color="#1e88e5"><b>💡 Hint</b></font></summary>

The feature axis disappears; batch and word remain.

</details>

In [ ]:
# Exercise 11.3: running this cell checks your answer
wrong_shape = ...  # your answer



<font size="4"><b>The same with names</b></font>

Average over the words with `einops.reduce` and the pattern from the explanation above.

<details><summary><font color="#1e88e5"><b>💡 Hint</b></font></summary>

`einops.reduce(T, "batch word feature -> batch feature", "mean")`.

</details>

In [ ]:
# Exercise 11.4: running this cell checks your answer
per_sentence_named = ...  # your answer



<font size="4"><b>Sum over the batch</b></font>

Which pattern sums over the batch? Use it, and get a block of shape (3, 4).

<details><summary><font color="#1e88e5"><b>💡 Hint</b></font></summary>

Leave `batch` out of the right side, and combine with `"sum"`.

</details>

In [ ]:
# Exercise 11.5: running this cell checks your answer
over_batch = ...  # your answer



<font size="4"><b>Loudest feature per word</b></font>

Give each word its largest feature value: reduce away `feature` with `"max"`.

<details><summary><font color="#1e88e5"><b>💡 Hint</b></font></summary>

`"batch word feature -> batch word"`.

</details>

In [ ]:
# Exercise 11.6: running this cell checks your answer
loudest = ...  # your answer



<font size="4"><b>Move the axes</b></font>

`einops.rearrange` moves axes without combining anything. Turn the block so words come first: `word batch feature`.

<details><summary><font color="#1e88e5"><b>💡 Hint</b></font></summary>

`einops.rearrange(T, "batch word feature -> word batch feature")`.

</details>

In [ ]:
# Exercise 11.7: running this cell checks your answer
words_first = ...  # your answer



<font size="4"><b>The names are the check</b></font>

Write `refuses(block)` that returns `True` when `einops.reduce(block, "batch word feature -> batch feature", "mean")` raises an `einops.EinopsError`, and `False` when it runs.

<details><summary><font color="#1e88e5"><b>💡 Hint</b></font></summary>

`try:` the reduce, `except einops.EinopsError: return True`, then `return False` after it.

</details>

In [ ]:
# Exercise 11.8: running this cell checks your answer
def refuses(block: np.ndarray) -> bool:
    ...  # your code here



<font size="4"><b>Named axes in PyTorch</b></font>

einops works on PyTorch tensors too. Average over the words of `torch.tensor(T)` with the same pattern.

<details><summary><font color="#1e88e5"><b>💡 Hint</b></font></summary>

Pass the tensor where you passed the array.

</details>

In [ ]:
# Exercise 11.9: running this cell checks your answer
per_sentence_torch = ...  # your answer



## <font color="#d84315">The courier, solved</font>

Back to the courier with 20 stops. The obvious state is the route so far. Take a depot O and three stops A, B, C. One route goes O, A, B, C and costs 9. Another goes O, B, A, C and costs 10. Both have visited the same three stops and both stand at C, so from here their futures are the same. The costlier one can never be part of the best tour.

So a better state holds two things: **which stops are done**, and **where the courier is now**. Every route that reaches the same state is merged, and only the cheapest is kept. Solving a problem by merging everything that reaches the same state, and keeping only the best, is called **dynamic programming**. For tours, the table of states is filled from states with one fewer stop done, and this version is known as the Held-Karp algorithm.

The plot compares the work: $n!$ orders against about $n^2 \times 2^n$ steps for the table.

In [ ]:
n_axis = np.arange(2, 21)
fig, ax = plt.subplots()
ax.semilogy(n_axis, [math.factorial(int(n)) for n in n_axis], "o-", color=PALETTE["pink"], label="every order: n!")
ax.semilogy(n_axis, [int(n) ** 2 * 2 ** int(n) for n in n_axis], "o-", color=PALETTE["mint"], label="the table: n^2 2^n")
ax.set_xlabel("stops n"); ax.set_ylabel("work (log scale)")
ax.set_title("Dynamic programming beats listing orders")
ax.legend()
plt.show()

pts8 = np.random.default_rng(11).uniform(0, 10, size=(9, 2))
D8 = np.sqrt(((pts8[:, None, :] - pts8[None, :, :]) ** 2).sum(axis=-1))
MERGE = {("O", "A"): 4, ("O", "B"): 3, ("A", "B"): 2, ("A", "C"): 5, ("B", "C"): 3}



<font size="4"><b>Two routes, one state</b></font>

Use your `route_cost` on `MERGE` to cost O, A, B, C and O, B, A, C. Give the cost that is kept after merging.

<details><summary><font color="#d84315"><b>💡 Hint</b></font></summary>

Both routes end in the same state, so keep the smaller cost.

</details>

In [ ]:
# Exercise 12.1: running this cell checks your answer
kept = ...  # your answer



<font size="4"><b>Count the states</b></font>

A state is a set of finished stops together with the current stop. Using the loose count from the video, $n \times 2^n$, how many states are there for 20 stops?

<details><summary><font color="#d84315"><b>💡 Hint</b></font></summary>

$2^{20}$ sets times 20 positions.

</details>

In [ ]:
# Exercise 12.2: running this cell checks your answer
states_20 = ...  # your answer



<font size="4"><b>The crossover</b></font>

Find the smallest number of stops $n$ for which $n^2 \times 2^n$ is less than $n!$.

<details><summary><font color="#d84315"><b>💡 Hint</b></font></summary>

Loop `n` upward from 2.

</details>

In [ ]:
# Exercise 12.3: running this cell checks your answer
crossover = ...  # your answer



<font size="4"><b>Dynamic programming for the courier</b></font>

Write `held_karp(D)` that returns the length of the shortest tour. The depot is point 0 and the stops are `1` to `n`. Keep a dictionary `best[(done, j)]`: the cheapest cost of leaving the depot, visiting exactly the stops in the frozenset `done`, and standing at stop `j` (which is in `done`).

<details><summary><font color="#d84315"><b>💡 Hint 1</b></font></summary>

Start with `best[(frozenset([j]), j)] = D[0][j]` for every stop `j`.

</details>

<details><summary><font color="#d84315"><b>💡 Hint 2</b></font></summary>

For each size from 2 to `n`, for each `subset` from `itertools.combinations(stops, size)`, for each `j` in it: `best[(done, j)]` is the minimum over `k` in `done` other than `j` of `best[(done - {j}, k)] + D[k][j]`.

</details>

<details><summary><font color="#d84315"><b>💡 Hint 3</b></font></summary>

At the end, close the loop: the minimum over `j` of `best[(all_stops, j)] + D[j][0]`.

</details>

In [ ]:
# Exercise 12.4: running this cell checks your answer
def held_karp(D: np.ndarray) -> float:
    ...  # your code here



<font size="4"><b>Eight stops, two ways</b></font>

`D8` is a courier with 8 stops. Find the shortest tour length with your `held_karp`, and again by brute force with `itertools.permutations` and your `tour_length`. Give `(held_karp_length, brute_force_length)`.

<details><summary><font color="#d84315"><b>💡 Hint</b></font></summary>

`min(tour_length(p, D8) for p in itertools.permutations(range(1, 9)))`. That checks 40,320 orders and takes a moment.

</details>

In [ ]:
# Exercise 12.5: running this cell checks your answer
hk8, brute8 = ..., ...  # your answer



<font size="4"><b>Does the output visit every stop?</b></font>

A reflex model could map features of the addresses to a route in one step, but nothing guarantees it visits every stop exactly once. Write `is_tour(order, n)` that returns `True` when `order` uses each of the stops `1` to `n` exactly once.

<details><summary><font color="#d84315"><b>💡 Hint</b></font></summary>

`sorted(order) == list(range(1, n + 1))`.

</details>

In [ ]:
# Exercise 12.6: running this cell checks your answer
def is_tour(order: list[int], n: int) -> bool:
    return ...  # your code here



<font size="4"><b>The variable-based version</b></font>

Model the 5-stop tour with one variable per position, each taking one of 5 stops. How many complete assignments are there, and how many pass the constraint that every stop appears once? Give a tuple.

<details><summary><font color="#d84315"><b>💡 Hint</b></font></summary>

5 values for each of 5 variables. The valid ones are the orders.

</details>

In [ ]:
# Exercise 12.7: running this cell checks your answer
pos_total, pos_valid = ..., ...  # your answer



<font size="4"><b>What the state remembers</b></font>

Which state is enough to plan the rest of the tour, with nothing extra? Answer `"a"`, `"b"` or `"c"`.

* a: the full route so far, in order
* b: the set of stops done, and the current stop
* c: only the current stop

<details><summary><font color="#d84315"><b>💡 Hint</b></font></summary>

The future depends on where she is and what is left. It does not depend on the order she took.

</details>

In [ ]:
# Exercise 12.8: running this cell checks your answer
state_choice = ...  # your answer



## <font color="#2e7d32">Writing a model down</font>

"Model this problem" is an instruction with a definite answer. Five lines pin a model down:

1. **In**: what comes in.
2. **Out**: what goes out.
3. **Unknown**: what has to be chosen or learned.
4. **Score**: how an answer is judged.
5. **Rung**: which rung of the ladder it sits on.

A state-based model has two checks of its own. **Sufficiency**: from the state alone, can you list the legal moves and what they cost? **Minimality**: can you delete a part of the state and still pass the first check?

Here is the courier's model written in the five lines.

In [ ]:
courier_spec = {
    "in": "20 addresses and the travel time between each pair",
    "out": "a tour",
    "unknown": "the order of the stops",
    "score": "total travel time, lower is better",
    "rung": "state (state: the stops done and the current stop)",
}
for k, v in courier_spec.items():
    print(f"{k:>8}: {v}")



<font size="4"><b>The spam filter&#x27;s five lines</b></font>

For the spam filter of section 3, give its rung (`"reflex"`, `"state"`, `"variable"` or `"logic"`) and how many numbers are unknown before learning.

<details><summary><font color="#2e7d32"><b>💡 Hint</b></font></summary>

Three weights and a bias.

</details>

In [ ]:
# Exercise 13.1: running this cell checks your answer
spam_rung, spam_unknowns = ..., ...  # your answer



<font size="4"><b>Twelve meetings</b></font>

Twelve meetings go into five rooms, and two meetings that overlap in time can't share a room. Give the rung, and the number of complete assignments of a room to every meeting before any constraint is applied.

<details><summary><font color="#2e7d32"><b>💡 Hint</b></font></summary>

Is there an order, or only a final assignment? Each of 12 variables has 5 values.

</details>

In [ ]:
# Exercise 13.2: running this cell checks your answer
meeting_rung, meeting_choices = ..., ...  # your answer



<font size="4"><b>The constraint as code</b></font>

A vague line such as "meetings shouldn't clash" can be built two ways. Write it over named variables. `rooms[i]` is the room of meeting `i`, and `overlaps` lists pairs of meetings that overlap in time. Write `rooms_ok(rooms, overlaps)`.

<details><summary><font color="#2e7d32"><b>💡 Hint</b></font></summary>

`all(rooms[i] != rooms[j] for i, j in overlaps)`.

</details>

In [ ]:
# Exercise 13.3: running this cell checks your answer
def rooms_ok(rooms: list[int], overlaps: list[tuple[int, int]]) -> bool:
    return ...  # your code here



<font size="4"><b>A small timetable</b></font>

Five meetings and three rooms, with the overlaps in `OVERLAPS`. Count the valid room choices with `itertools.product` and your `rooms_ok`.

<details><summary><font color="#2e7d32"><b>💡 Hint</b></font></summary>

`itertools.product(range(3), repeat=5)` gives every choice. Count the ones that pass.

</details>

In [ ]:
# Exercise 13.4: running this cell checks your answer
OVERLAPS = [(0, 1), (1, 2), (2, 3), (3, 4), (0, 2)]
n_timetables = ...  # your answer



<font size="4"><b>Find the vague line</b></font>

The usual mistake is a line left vague, so that two people would build two different models from it. Which key of `spam_draft` is vague? Answer with the key as a string.

<details><summary><font color="#2e7d32"><b>💡 Hint</b></font></summary>

Section 3 named exactly which measurements of the email the filter uses. Does this draft?

</details>

In [ ]:
# Exercise 13.5: running this cell checks your answer
spam_draft = {"in": "an email, turned into some features", "out": "spam or not spam",
              "unknown": "3 weights and a bias", "score": "squared error on labelled emails", "rung": "reflex"}
vague_key = ...  # your answer



<font size="4"><b>Sufficiency</b></font>

The state "only the current stop" fails sufficiency. Which question can't it answer? Answer `"a"` or `"b"`.

* a: which stops can she go to next?
* b: how far is it from here to stop 3?

<details><summary><font color="#2e7d32"><b>💡 Hint</b></font></summary>

The distance table answers one of these from the current stop alone.

</details>

In [ ]:
# Exercise 13.6: running this cell checks your answer
cannot_answer = ...  # your answer



<font size="4"><b>Minimality</b></font>

The full route so far passes sufficiency and fails minimality. For 12 stops, count the states of each kind: every ordered route of 1 to 12 distinct stops, against $n \times 2^n$ for (stops done, current stop). Give a tuple.

<details><summary><font color="#2e7d32"><b>💡 Hint</b></font></summary>

The ordered routes of length $k$ number `math.perm(12, k)`. Add them for $k = 1, \dots, 12$.

</details>

In [ ]:
# Exercise 13.7: running this cell checks your answer
route_states, set_states = ..., ...  # your answer



<font size="4"><b>A logic model&#x27;s lines</b></font>

On the logic rung, the lines become the symbols, what each one means, and the rules. Write the rain example's rules as a list of `(if_name, then_name)` pairs, the format your `forward_chain` takes.

<details><summary><font color="#2e7d32"><b>💡 Hint</b></font></summary>

Two rules: Rain to Wet, and Wet to Slippery.

</details>

In [ ]:
# Exercise 13.8: running this cell checks your answer
rain_rules = ...  # your answer



## <font color="#5e35b1">What you built</font>

* The recipe: modeling writes the problem down, inference answers questions with the model, and learning fills in its numbers from data. Keeping them separate lets one algorithm serve many problems.
* The ladder: reflex, state-based, variable-based and logic-based models. Each rung describes more problems and costs more computation, so pick the lowest rung that fits.
* The notation: vectors and matrices with shapes that must line up, sigma and pi, logs for products of probabilities, and named axes with einops.
* Learning: a loss written as a sigma, its gradient, and gradient descent by hand and in PyTorch.
* The courier: dynamic programming, with a state that drops the order of finished stops, turns 77 years of brute force into under half a second.

The next lesson shows how gradients are computed for any model, however large, by passing numbers backwards through a graph.



Run the cell below any time to see how far you are.

In [ ]:
progress()

<font size="2" color="#8a84b3">© 2026 Applied Data Science (<a href="https://www.youtube.com/@neuralmodules" style="color:#ff8fc4">youtube.com/@neuralmodules</a>). All rights reserved. This notebook and the lesson video belong to the channel. They are for personal, non-commercial study. You may run and change your own copy. You may not republish, share, sell or reproduce this notebook, the videos or any part of them without written permission from the channel.</font>